# Homework 3 — Dataset 2: NYC 311 Service Requests

## Kernel and Non-Kernel Learning Under Real-World Data Imperfections

This notebook implements the full pipeline for **Dataset 2: NYC 311 Service Requests**.

The assignment requires two prediction tasks for this dataset:

1. **Complaint Category Classification**  
   Predict the complaint category from available request information.

2. **Resolution Time Regression**  
   Predict how long it takes to close or resolve a 311 request.

The dataset-specific engineering requirements are:

- Intelligent sampling while preserving distribution integrity
- Duplicate report resolution
- Meaningful temporal feature engineering
- Missing value investigation
- Outlier analysis with Z-score, IQR, and Isolation Forest
- Feature quality analysis including leakage, redundancy, and near-constant features
- From-scratch implementation of non-kernel and kernel models
- Failure analysis and research-level discussion

## Section 0 — Project Setup

We import only general-purpose scientific Python libraries.

The core machine learning models are implemented from scratch using NumPy. We do not use scikit-learn for model implementation.

In [1]:
# ============================================================
# Section 0: Project Setup
# ============================================================

import os
import re
import gc
import zipfile
import time
import math
import warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 160)
pd.set_option("display.width", 220)

OUTPUT_DIR = Path("outputs_nyc311")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Libraries imported successfully.")
print(f"Random seed fixed at: {RANDOM_STATE}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


Libraries imported successfully.
Random seed fixed at: 42
Output directory: C:\Users\Administrator\Desktop\ML Projects\outputs_nyc311


## Section 1 — Locate and Inspect the NYC 311 ZIP File

The uploaded file is a compressed CSV file.

We first locate the ZIP file and inspect the internal CSV file name before reading data.

In [ ]:
# ============================================================
# Section 1: Locate and Inspect the NYC 311 ZIP File
# ============================================================

# Robust ZIP discovery: works in local notebook folders and in /mnt/data.
# It supports exact uploaded names as well as any NYC 311 ZIP whose name starts with 311_Service_Requests.
explicit_zip_candidates = [
    Path("311_Service_Requests_from_2020_to_Present_20260618.zip"),
    Path("./311_Service_Requests_from_2020_to_Present_20260618.zip"),
    Path("../311_Service_Requests_from_2020_to_Present_20260618.zip"),
    Path("/mnt/data/311_Service_Requests_from_2020_to_Present_20260618.zip"),
]

glob_zip_candidates = []
for folder in [Path("."), Path(".."), Path("/mnt/data")]:
    if folder.exists():
        glob_zip_candidates.extend(sorted(folder.glob("311_Service_Requests*.zip")))
        glob_zip_candidates.extend(sorted(folder.glob("*311*Service*Requests*.zip")))

ZIP_CANDIDATES = list(dict.fromkeys(explicit_zip_candidates + glob_zip_candidates))

ZIP_PATH = None
for candidate in ZIP_CANDIDATES:
    if candidate.exists():
        ZIP_PATH = candidate
        break

if ZIP_PATH is None:
    searched = "\n".join(str(p) for p in ZIP_CANDIDATES)
    raise FileNotFoundError(
        "NYC 311 ZIP file was not found. Put the ZIP beside this notebook or update ZIP_CANDIDATES.\n"
        f"Searched paths:\n{searched}"
    )

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    zip_files = z.namelist()
    csv_files = [name for name in zip_files if name.lower().endswith(".csv")]
    if len(csv_files) == 0:
        raise ValueError(f"No CSV file found inside ZIP. ZIP contents: {zip_files[:20]}")
    INTERNAL_CSV_NAME = csv_files[0]
    internal_info = z.getinfo(INTERNAL_CSV_NAME)

print("ZIP file found:", ZIP_PATH)
print("Internal CSV:", INTERNAL_CSV_NAME)
print(f"Compressed ZIP size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")
print(f"Internal CSV size: {internal_info.file_size / (1024**2):.2f} MB")


## Section 2 — Read a Small Preview and Build the Initial Schema

The full NYC 311 dataset is large, so we begin by reading only a small preview.

This lets us understand:

- Column names
- Data types inferred by pandas
- Example values
- Candidate targets
- Date formats

In [ ]:
# ============================================================
# Section 2: Read Preview and Initial Schema
# ============================================================

preview_df = pd.read_csv(
    ZIP_PATH,
    compression="zip",
    nrows=10,
    low_memory=False
)

print("Preview shape:", preview_df.shape)
print("Columns:")
for i, col in enumerate(preview_df.columns, start=1):
    print(f"{i:02d}. {col}")

display(preview_df.head())

initial_schema_table = pd.DataFrame({
    "column": preview_df.columns,
    "preview_dtype": preview_df.dtypes.astype(str).values,
    "preview_non_null_count": preview_df.notna().sum().values,
    "preview_unique_values": preview_df.nunique(dropna=False).values
})

display(initial_schema_table)


## Section 3 — Dataset-Specific Target Definition

For Dataset 2, the assignment defines two targets:

### Classification Target

`Problem (formerly Complaint Type)`

This is the complaint category. We predict the category of a 311 service request.

### Regression Target

`resolution_time_hours`

This is engineered from:

`Closed Date - Created Date`

The target is measured in hours. Requests without a valid `Closed Date` cannot be used for supervised resolution-time regression, but they remain useful for data-quality analysis.

In [ ]:
# ============================================================
# Section 3: Dataset-Specific Target Definition
# ============================================================

UNIQUE_KEY_COL = "Unique Key"
CREATED_DATE_COL = "Created Date"
CLOSED_DATE_COL = "Closed Date"
CLASSIFICATION_TARGET_COL = "Problem (formerly Complaint Type)"
REGRESSION_TARGET_COL = "resolution_time_hours"

required_columns = [UNIQUE_KEY_COL, CREATED_DATE_COL, CLOSED_DATE_COL, CLASSIFICATION_TARGET_COL]
missing_required = [col for col in required_columns if col not in preview_df.columns]

if missing_required:
    raise ValueError(f"Required columns are missing from the dataset: {missing_required}")

print("Target columns verified.")
print("Classification target:", CLASSIFICATION_TARGET_COL)
print("Regression target to be engineered:", REGRESSION_TARGET_COL)


## Section 4 — Chunked First-Pass Profiling

The full CSV is large. Reading it all into memory before understanding its structure is risky.

In this first pass, we read the dataset in chunks and compute important global summaries:

- Complaint category distribution
- Status distribution
- Borough distribution
- Agency distribution
- Open data channel distribution
- Missing date counts
- Duplicate `Unique Key` counts

This supports intelligent sampling and duplicate investigation.

In [ ]:
# ============================================================
# Section 4: Chunked First-Pass Profiling
# ============================================================

CHUNKSIZE = 100_000

FIRST_PASS_USECOLS = [
    UNIQUE_KEY_COL,
    CREATED_DATE_COL,
    CLOSED_DATE_COL,
    CLASSIFICATION_TARGET_COL,
    "Problem Detail (formerly Descriptor)",
    "Agency",
    "Agency Name",
    "Status",
    "Borough",
    "Incident Zip",
    "City",
    "Open Data Channel Type",
    "Location Type",
    "Latitude",
    "Longitude"
]
FIRST_PASS_USECOLS = [col for col in FIRST_PASS_USECOLS if col in preview_df.columns]

category_counts = Counter()
status_counts = Counter()
borough_counts = Counter()
agency_counts = Counter()
channel_counts = Counter()

seen_unique_keys = set()
duplicate_unique_key_count = 0
rows_total = 0
created_missing_count = 0
closed_missing_count = 0
classification_target_missing_count = 0

start_time = time.time()

for chunk_idx, chunk in enumerate(pd.read_csv(
    ZIP_PATH,
    compression="zip",
    chunksize=CHUNKSIZE,
    usecols=FIRST_PASS_USECOLS,
    low_memory=False
)):
    rows_total += len(chunk)

    category_counts.update(chunk[CLASSIFICATION_TARGET_COL].fillna("__MISSING__"))

    if "Status" in chunk.columns:
        status_counts.update(chunk["Status"].fillna("__MISSING__"))
    if "Borough" in chunk.columns:
        borough_counts.update(chunk["Borough"].fillna("__MISSING__"))
    if "Agency" in chunk.columns:
        agency_counts.update(chunk["Agency"].fillna("__MISSING__"))
    if "Open Data Channel Type" in chunk.columns:
        channel_counts.update(chunk["Open Data Channel Type"].fillna("__MISSING__"))

    created_missing_count += chunk[CREATED_DATE_COL].isna().sum()
    closed_missing_count += chunk[CLOSED_DATE_COL].isna().sum()
    classification_target_missing_count += chunk[CLASSIFICATION_TARGET_COL].isna().sum()

    if UNIQUE_KEY_COL in chunk.columns:
        keys = chunk[UNIQUE_KEY_COL].dropna().astype(str).values
        for key in keys:
            if key in seen_unique_keys:
                duplicate_unique_key_count += 1
            else:
                seen_unique_keys.add(key)

    if (chunk_idx + 1) % 10 == 0:
        print(f"Processed chunks: {chunk_idx + 1}, rows so far: {rows_total:,}")

first_pass_runtime = time.time() - start_time

first_pass_summary = pd.DataFrame({
    "metric": [
        "rows_total",
        "created_date_missing_count",
        "closed_date_missing_count",
        "classification_target_missing_count",
        "duplicate_unique_key_count",
        "first_pass_runtime_seconds"
    ],
    "value": [
        rows_total,
        created_missing_count,
        closed_missing_count,
        classification_target_missing_count,
        duplicate_unique_key_count,
        first_pass_runtime
    ]
})

display(first_pass_summary)

category_distribution_table = pd.DataFrame(category_counts.items(), columns=["complaint_category", "count"])
category_distribution_table = category_distribution_table.sort_values("count", ascending=False).reset_index(drop=True)
category_distribution_table["rate"] = category_distribution_table["count"] / rows_total

display(category_distribution_table.head(40))

status_distribution_table = pd.DataFrame(status_counts.items(), columns=["status", "count"]).sort_values("count", ascending=False).reset_index(drop=True)
status_distribution_table["rate"] = status_distribution_table["count"] / rows_total

display(status_distribution_table.head(20))


## Section 5 — Intelligent Stratified Sampling

The NYC 311 dataset is very large. Kernel methods and from-scratch algorithms cannot be trained directly on millions of records because many kernel methods require an `O(n²)` kernel matrix.

Therefore, we create an intelligent sample.

The sampling strategy preserves complaint-category distribution by:

1. Selecting the most frequent complaint categories.
2. Computing class-specific quotas.
3. Applying streaming reservoir sampling per complaint category.
4. Keeping a reproducible random sample.

This satisfies the requirement to sample intelligently while maintaining distribution integrity.

In [ ]:
# ============================================================
# Section 5: Intelligent Stratified Sampling
# ============================================================

TOP_N_COMPLAINT_CLASSES = 20
MAX_SAMPLE_ROWS = 120_000
MIN_ROWS_PER_SELECTED_CLASS = 1_000

valid_category_distribution = category_distribution_table[
    category_distribution_table["complaint_category"] != "__MISSING__"
].copy()

selected_categories = valid_category_distribution.head(TOP_N_COMPLAINT_CLASSES)["complaint_category"].tolist()
selected_category_counts = dict(
    zip(valid_category_distribution["complaint_category"], valid_category_distribution["count"])
)

def compute_sampling_quotas(selected_categories, global_counts, max_rows, min_rows_per_class):
    total_selected = sum(global_counts[c] for c in selected_categories)
    raw_quotas = {}

    for c in selected_categories:
        proportional_quota = int(round(max_rows * global_counts[c] / total_selected))
        quota = max(min_rows_per_class, proportional_quota)
        quota = min(quota, global_counts[c])
        raw_quotas[c] = quota

    # If quotas exceed max_rows, scale down proportionally while preserving at least 1 row.
    total_quota = sum(raw_quotas.values())
    if total_quota > max_rows:
        scale = max_rows / total_quota
        scaled = {c: max(1, int(raw_quotas[c] * scale)) for c in selected_categories}
        # Fix rounding deficit/excess
        while sum(scaled.values()) < max_rows:
            candidates = sorted(selected_categories, key=lambda c: global_counts[c], reverse=True)
            for c in candidates:
                if scaled[c] < global_counts[c] and sum(scaled.values()) < max_rows:
                    scaled[c] += 1
        while sum(scaled.values()) > max_rows:
            candidates = sorted(selected_categories, key=lambda c: scaled[c], reverse=True)
            for c in candidates:
                if scaled[c] > 1 and sum(scaled.values()) > max_rows:
                    scaled[c] -= 1
        return scaled

    return raw_quotas

sampling_quotas = compute_sampling_quotas(
    selected_categories=selected_categories,
    global_counts=selected_category_counts,
    max_rows=MAX_SAMPLE_ROWS,
    min_rows_per_class=MIN_ROWS_PER_SELECTED_CLASS
)

sampling_quota_table = pd.DataFrame({
    "complaint_category": list(sampling_quotas.keys()),
    "global_count": [selected_category_counts[c] for c in sampling_quotas.keys()],
    "sample_quota": [sampling_quotas[c] for c in sampling_quotas.keys()]
})
sampling_quota_table["global_rate_within_selected"] = sampling_quota_table["global_count"] / sampling_quota_table["global_count"].sum()
sampling_quota_table["sample_rate_within_selected"] = sampling_quota_table["sample_quota"] / sampling_quota_table["sample_quota"].sum()
sampling_quota_table["absolute_rate_difference"] = (sampling_quota_table["global_rate_within_selected"] - sampling_quota_table["sample_rate_within_selected"]).abs()

display(sampling_quota_table)

# Streaming reservoir by random priority per class.
rng = np.random.default_rng(RANDOM_STATE)
reservoirs = {c: pd.DataFrame() for c in selected_categories}
priority_col = "__random_priority__"

start_time = time.time()

for chunk_idx, chunk in enumerate(pd.read_csv(
    ZIP_PATH,
    compression="zip",
    chunksize=CHUNKSIZE,
    low_memory=False
)):
    chunk = chunk[chunk[CLASSIFICATION_TARGET_COL].isin(selected_categories)].copy()

    if len(chunk) == 0:
        continue

    chunk[priority_col] = rng.random(len(chunk))

    for category, quota in sampling_quotas.items():
        part = chunk[chunk[CLASSIFICATION_TARGET_COL] == category]
        if len(part) == 0:
            continue
        combined = pd.concat([reservoirs[category], part], axis=0, ignore_index=True)
        combined = combined.nsmallest(quota, priority_col)
        reservoirs[category] = combined

    if (chunk_idx + 1) % 10 == 0:
        current_size = sum(len(v) for v in reservoirs.values())
        print(f"Processed chunks: {chunk_idx + 1}, current reservoir rows: {current_size:,}")

nyc311_sample_raw = pd.concat(reservoirs.values(), axis=0, ignore_index=True)
nyc311_sample_raw = nyc311_sample_raw.drop(columns=[priority_col], errors="ignore")
nyc311_sample_raw = nyc311_sample_raw.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

sampling_runtime = time.time() - start_time

print("Final sampled dataframe shape:", nyc311_sample_raw.shape)
print(f"Sampling runtime: {sampling_runtime:.2f} seconds")

sample_category_distribution = nyc311_sample_raw[CLASSIFICATION_TARGET_COL].value_counts().reset_index()
sample_category_distribution.columns = ["complaint_category", "sample_count"]
sample_category_distribution["sample_rate"] = sample_category_distribution["sample_count"] / len(nyc311_sample_raw)

display(sample_category_distribution)

nyc311_sample_raw.to_csv(OUTPUT_DIR / "nyc311_intelligent_stratified_sample_raw.csv", index=False)


## Section 6 — Sampling Integrity Check

A sampled dataset is useful only if it preserves the main distributional structure of the original dataset.

We compare the selected complaint-class distribution in the original data and the sample.

A small absolute difference means the sampling procedure preserved distribution integrity.

In [ ]:
# ============================================================
# Section 6: Sampling Integrity Check
# ============================================================

original_selected_distribution = valid_category_distribution[
    valid_category_distribution["complaint_category"].isin(selected_categories)
].copy()
original_selected_distribution = original_selected_distribution[["complaint_category", "count"]]
original_selected_distribution = original_selected_distribution.rename(columns={"count": "original_count"})
original_selected_distribution["original_rate_within_selected"] = (
    original_selected_distribution["original_count"] / original_selected_distribution["original_count"].sum()
)

sample_selected_distribution = nyc311_sample_raw[CLASSIFICATION_TARGET_COL].value_counts().reset_index()
sample_selected_distribution.columns = ["complaint_category", "sample_count"]
sample_selected_distribution["sample_rate_within_selected"] = (
    sample_selected_distribution["sample_count"] / sample_selected_distribution["sample_count"].sum()
)

sampling_integrity_table = original_selected_distribution.merge(
    sample_selected_distribution,
    on="complaint_category",
    how="left"
)
sampling_integrity_table["absolute_rate_difference"] = (
    sampling_integrity_table["original_rate_within_selected"] - sampling_integrity_table["sample_rate_within_selected"]
).abs()

sampling_integrity_table = sampling_integrity_table.sort_values("absolute_rate_difference", ascending=False).reset_index(drop=True)

display(sampling_integrity_table)

print("Mean absolute distribution difference:", sampling_integrity_table["absolute_rate_difference"].mean())
print("Max absolute distribution difference:", sampling_integrity_table["absolute_rate_difference"].max())

sampling_integrity_table.to_csv(OUTPUT_DIR / "nyc311_sampling_integrity_table.csv", index=False)


## Section 7 — Duplicate Report Resolution

The assignment explicitly requires resolving duplicate reports.

We investigate two types of duplicates:

1. **Exact Unique Key duplicates**  
   The `Unique Key` should be unique. Repeated keys indicate duplicated records.

2. **Near-duplicate service reports**  
   Two reports may have different unique keys but refer to the same incident if they share similar complaint type, descriptor, address, borough, and creation time window.

We do not blindly delete all near-duplicates. We document them and remove exact duplicates first. Then we create a near-duplicate cluster identifier that can be used in analysis.

In [ ]:
# ============================================================
# Section 7: Duplicate Report Resolution
# ============================================================

nyc311 = nyc311_sample_raw.copy()

exact_duplicate_rows = nyc311.duplicated().sum()
unique_key_duplicate_rows = nyc311[UNIQUE_KEY_COL].duplicated().sum() if UNIQUE_KEY_COL in nyc311.columns else np.nan

print("Exact duplicated rows:", exact_duplicate_rows)
print("Duplicated Unique Key rows:", unique_key_duplicate_rows)

# Resolve exact Unique Key duplicates by keeping the first record.
before_dedup_rows = len(nyc311)
if UNIQUE_KEY_COL in nyc311.columns:
    nyc311 = nyc311.drop_duplicates(subset=[UNIQUE_KEY_COL], keep="first").reset_index(drop=True)
else:
    nyc311 = nyc311.drop_duplicates().reset_index(drop=True)
after_dedup_rows = len(nyc311)

print("Rows before exact duplicate resolution:", before_dedup_rows)
print("Rows after exact duplicate resolution:", after_dedup_rows)
print("Removed rows:", before_dedup_rows - after_dedup_rows)

# Parse Created Date for near-duplicate windows.
nyc311["created_datetime"] = pd.to_datetime(nyc311[CREATED_DATE_COL], errors="coerce")
nyc311["created_hour_window"] = nyc311["created_datetime"].dt.floor("H")

near_duplicate_components = []
for col in [CLASSIFICATION_TARGET_COL, "Problem Detail (formerly Descriptor)", "Incident Address", "Borough", "Incident Zip", "created_hour_window"]:
    if col in nyc311.columns:
        near_duplicate_components.append(col)

if len(near_duplicate_components) >= 3:
    nyc311["near_duplicate_key"] = nyc311[near_duplicate_components].astype(str).agg("|".join, axis=1)
    near_duplicate_counts = nyc311["near_duplicate_key"].value_counts()
    nyc311["near_duplicate_cluster_size"] = nyc311["near_duplicate_key"].map(near_duplicate_counts)
else:
    nyc311["near_duplicate_key"] = "not_available"
    nyc311["near_duplicate_cluster_size"] = 1

near_duplicate_summary = pd.DataFrame({
    "metric": [
        "rows_after_exact_dedup",
        "rows_in_near_duplicate_clusters_size_gt_1",
        "near_duplicate_cluster_count_size_gt_1",
        "max_near_duplicate_cluster_size"
    ],
    "value": [
        len(nyc311),
        int((nyc311["near_duplicate_cluster_size"] > 1).sum()),
        int((near_duplicate_counts > 1).sum()) if "near_duplicate_counts" in locals() else 0,
        int(nyc311["near_duplicate_cluster_size"].max())
    ]
})

display(near_duplicate_summary)

display(
    nyc311.sort_values("near_duplicate_cluster_size", ascending=False)
    .head(20)[[col for col in [UNIQUE_KEY_COL, CREATED_DATE_COL, CLASSIFICATION_TARGET_COL, "Problem Detail (formerly Descriptor)", "Incident Address", "Borough", "Incident Zip", "near_duplicate_cluster_size"] if col in nyc311.columns]]
)

near_duplicate_summary.to_csv(OUTPUT_DIR / "nyc311_duplicate_resolution_summary.csv", index=False)
nyc311.to_csv(OUTPUT_DIR / "nyc311_after_duplicate_resolution.csv", index=False)


## Section 8 — Target Engineering: Resolution Time

Resolution time is the regression target.

It is computed as:

`Closed Date - Created Date`

in hours.

We also create flags for problematic cases:

- Missing created date
- Missing closed date
- Negative resolution time
- Extremely long resolution time

These cases are important for label quality analysis.

In [ ]:
# ============================================================
# Section 8: Target Engineering: Resolution Time
# ============================================================

nyc311["created_datetime"] = pd.to_datetime(nyc311[CREATED_DATE_COL], errors="coerce")
nyc311["closed_datetime"] = pd.to_datetime(nyc311[CLOSED_DATE_COL], errors="coerce")

nyc311[REGRESSION_TARGET_COL] = (
    nyc311["closed_datetime"] - nyc311["created_datetime"]
).dt.total_seconds() / 3600.0

nyc311["target_missing_created_date"] = nyc311["created_datetime"].isna().astype(int)
nyc311["target_missing_closed_date"] = nyc311["closed_datetime"].isna().astype(int)
nyc311["target_negative_resolution_time"] = (nyc311[REGRESSION_TARGET_COL] < 0).fillna(False).astype(int)
nyc311["target_extreme_resolution_time_gt_365_days"] = (nyc311[REGRESSION_TARGET_COL] > 365 * 24).fillna(False).astype(int)

nyc311["valid_resolution_time_label"] = (
    nyc311[REGRESSION_TARGET_COL].notna()
    & np.isfinite(nyc311[REGRESSION_TARGET_COL])
    & (nyc311[REGRESSION_TARGET_COL] >= 0)
    & (nyc311[REGRESSION_TARGET_COL] <= 365 * 24)
).astype(int)

resolution_target_summary = pd.DataFrame({
    "metric": [
        "rows_total",
        "valid_resolution_time_labels",
        "missing_created_date",
        "missing_closed_date",
        "negative_resolution_time",
        "resolution_time_gt_365_days"
    ],
    "value": [
        len(nyc311),
        int(nyc311["valid_resolution_time_label"].sum()),
        int(nyc311["target_missing_created_date"].sum()),
        int(nyc311["target_missing_closed_date"].sum()),
        int(nyc311["target_negative_resolution_time"].sum()),
        int(nyc311["target_extreme_resolution_time_gt_365_days"].sum())
    ]
})

display(resolution_target_summary)

print("Resolution time summary for valid labels:")
display(nyc311.loc[nyc311["valid_resolution_time_label"] == 1, REGRESSION_TARGET_COL].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

resolution_target_summary.to_csv(OUTPUT_DIR / "nyc311_resolution_target_summary.csv", index=False)


## Section 9 — Temporal Feature Engineering

NYC 311 requests are time-dependent.

We create temporal features from `Created Date` only because it is available at prediction time.

The features include:

- Year
- Month
- Day of week
- Hour
- Weekend flag
- After-hours flag
- Season
- Cyclical hour encoding
- Cyclical month encoding

We avoid using `Closed Date` as an input feature because it directly defines the resolution-time target and would cause leakage.

In [ ]:
# ============================================================
# Section 9: Temporal Feature Engineering
# ============================================================

def add_temporal_features(data, created_col="created_datetime"):
    df_time = data.copy()
    dt = df_time[created_col]

    df_time["created_year"] = dt.dt.year
    df_time["created_month"] = dt.dt.month
    df_time["created_day"] = dt.dt.day
    df_time["created_dayofweek"] = dt.dt.dayofweek
    df_time["created_hour"] = dt.dt.hour
    df_time["created_quarter"] = dt.dt.quarter
    df_time["created_is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(float)
    df_time["created_is_after_hours"] = ((dt.dt.hour < 8) | (dt.dt.hour >= 18)).astype(float)

    # Season encoding: winter=0, spring=1, summer=2, fall=3
    month = dt.dt.month
    season = np.select(
        [month.isin([12, 1, 2]), month.isin([3, 4, 5]), month.isin([6, 7, 8]), month.isin([9, 10, 11])],
        [0, 1, 2, 3],
        default=np.nan
    )
    df_time["created_season"] = season

    df_time["created_hour_sin"] = np.sin(2 * np.pi * df_time["created_hour"] / 24)
    df_time["created_hour_cos"] = np.cos(2 * np.pi * df_time["created_hour"] / 24)
    df_time["created_month_sin"] = np.sin(2 * np.pi * df_time["created_month"] / 12)
    df_time["created_month_cos"] = np.cos(2 * np.pi * df_time["created_month"] / 12)

    return df_time

nyc311 = add_temporal_features(nyc311)

temporal_features = [
    "created_year",
    "created_month",
    "created_day",
    "created_dayofweek",
    "created_hour",
    "created_quarter",
    "created_is_weekend",
    "created_is_after_hours",
    "created_season",
    "created_hour_sin",
    "created_hour_cos",
    "created_month_sin",
    "created_month_cos"
]

temporal_feature_summary = nyc311[temporal_features].describe().T

display(temporal_feature_summary)

# Visualize request volume by hour and day of week
plt.figure(figsize=(9, 4))
nyc311["created_hour"].value_counts().sort_index().plot(kind="bar")
plt.title("NYC 311 Sample Request Volume by Hour")
plt.xlabel("Created Hour")
plt.ylabel("Count")
plt.grid(axis="y", alpha=0.3)
plt.show()

plt.figure(figsize=(9, 4))
nyc311["created_dayofweek"].value_counts().sort_index().plot(kind="bar")
plt.title("NYC 311 Sample Request Volume by Day of Week")
plt.xlabel("Day of Week, Monday=0")
plt.ylabel("Count")
plt.grid(axis="y", alpha=0.3)
plt.show()


## Section 10 — Geographic and Channel Feature Engineering

NYC 311 requests are geographically structured.

We create interpretable geographic features:

- Whether coordinates exist
- Rounded coordinate grid features
- Borough and ZIP categorical features

We also preserve channel information because complaint reporting behavior may differ between phone, mobile, online, and other channels.

In [ ]:
# ============================================================
# Section 10: Geographic and Channel Feature Engineering
# ============================================================

for col in ["Latitude", "Longitude"]:
    if col in nyc311.columns:
        nyc311[col] = pd.to_numeric(nyc311[col], errors="coerce")

if "Latitude" in nyc311.columns and "Longitude" in nyc311.columns:
    nyc311["has_geo_coordinates"] = (nyc311["Latitude"].notna() & nyc311["Longitude"].notna()).astype(int)
    nyc311["latitude_rounded_2"] = nyc311["Latitude"].round(2)
    nyc311["longitude_rounded_2"] = nyc311["Longitude"].round(2)
else:
    nyc311["has_geo_coordinates"] = 0

# Normalize common categorical columns.
for col in ["Borough", "City", "Incident Zip", "Open Data Channel Type", "Location Type", "Agency", "Agency Name"]:
    if col in nyc311.columns:
        nyc311[col] = nyc311[col].fillna("Unknown").astype(str).str.strip()
        nyc311[col] = nyc311[col].replace({"nan": "Unknown", "": "Unknown"})

geo_channel_summary = pd.DataFrame({
    "column": [col for col in ["Borough", "City", "Incident Zip", "Open Data Channel Type", "Location Type", "has_geo_coordinates"] if col in nyc311.columns],
    "missing_rate": [nyc311[col].isna().mean() for col in ["Borough", "City", "Incident Zip", "Open Data Channel Type", "Location Type", "has_geo_coordinates"] if col in nyc311.columns],
    "unique_values": [nyc311[col].nunique(dropna=False) for col in ["Borough", "City", "Incident Zip", "Open Data Channel Type", "Location Type", "has_geo_coordinates"] if col in nyc311.columns]
})

display(geo_channel_summary)

if "Borough" in nyc311.columns:
    display(nyc311["Borough"].value_counts(dropna=False).head(20))

if "Open Data Channel Type" in nyc311.columns:
    display(nyc311["Open Data Channel Type"].value_counts(dropna=False).head(20))


## Section 11 — Missing Value Analysis

The assignment requires a rigorous missing data investigation.

We compute:

- Missing count
- Missing rate
- Data type
- Unique values
- Missingness group

We later use this table to design imputation strategies and identify MCAR/MAR/MNAR mechanisms.

In [ ]:
# ============================================================
# Section 11: Missing Value Analysis
# ============================================================

def missing_group(rate):
    if rate == 0:
        return "No missing"
    elif rate <= 0.05:
        return "Low missingness"
    elif rate <= 0.30:
        return "Moderate missingness"
    elif rate < 1.00:
        return "High missingness"
    else:
        return "Completely missing"

missing_summary = pd.DataFrame({
    "column": nyc311.columns,
    "dtype": nyc311.dtypes.astype(str).values,
    "missing_count": nyc311.isna().sum().values,
    "missing_rate": nyc311.isna().mean().values,
    "non_missing_count": nyc311.notna().sum().values,
    "unique_values": nyc311.nunique(dropna=False).values
})
missing_summary["missing_group"] = missing_summary["missing_rate"].apply(missing_group)
missing_summary = missing_summary.sort_values(["missing_rate", "unique_values"], ascending=[False, True]).reset_index(drop=True)

display(missing_summary)

display(missing_summary["missing_group"].value_counts())

plt.figure(figsize=(12, 8))
top_missing = missing_summary[missing_summary["missing_rate"] > 0].head(35)
plt.barh(top_missing["column"][::-1], top_missing["missing_rate"][::-1])
plt.title("NYC 311 Top Missing Value Rates")
plt.xlabel("Missing Rate")
plt.ylabel("Column")
plt.grid(axis="x", alpha=0.3)
plt.show()

missing_summary.to_csv(OUTPUT_DIR / "nyc311_missing_summary.csv", index=False)


## Section 12 — Missingness Indicators and MCAR/MAR/MNAR Evidence

We create missingness indicators for important columns and test whether missingness is associated with observed variables.

If missingness is associated with other observed variables, it is more consistent with **MAR** than **MCAR**.

Some columns are marked as **Suspected MNAR** based on domain reasoning, especially when missingness may depend on hidden reporting behavior or unobserved incident characteristics.

In [ ]:
# ============================================================
# Section 12: Missingness Indicators and MCAR/MAR/MNAR Evidence
# ============================================================

important_missingness_columns = [
    CREATED_DATE_COL,
    CLOSED_DATE_COL,
    CLASSIFICATION_TARGET_COL,
    "Problem Detail (formerly Descriptor)",
    "Additional Details",
    "Location Type",
    "Incident Zip",
    "Incident Address",
    "Cross Street 1",
    "Cross Street 2",
    "Resolution Description",
    "Due Date",
    "Resolution Action Updated Date",
    "Latitude",
    "Longitude",
    "Borough",
    "Open Data Channel Type"
]
important_missingness_columns = [col for col in important_missingness_columns if col in nyc311.columns]

missing_indicator_columns = []
for col in important_missingness_columns:
    if nyc311[col].isna().sum() > 0:
        indicator_col = f"is_missing__{col}"
        indicator_col = indicator_col.replace(" ", "_").replace("(", "").replace(")", "")
        nyc311[indicator_col] = nyc311[col].isna().astype(int)
        missing_indicator_columns.append(indicator_col)

print("Missingness indicators created:", len(missing_indicator_columns))
print(missing_indicator_columns)

candidate_numeric_explainers = [
    "created_year", "created_month", "created_dayofweek", "created_hour",
    "created_is_weekend", "created_is_after_hours", "Latitude", "Longitude",
    "has_geo_coordinates", "near_duplicate_cluster_size", REGRESSION_TARGET_COL
]
candidate_numeric_explainers = [col for col in candidate_numeric_explainers if col in nyc311.columns and pd.api.types.is_numeric_dtype(nyc311[col])]

candidate_categorical_explainers = [
    CLASSIFICATION_TARGET_COL, "Agency", "Status", "Borough", "City", "Incident Zip",
    "Open Data Channel Type", "Location Type"
]
candidate_categorical_explainers = [col for col in candidate_categorical_explainers if col in nyc311.columns]


def mann_whitney_missingness_test(data, missing_col, numeric_col, min_group_size=20):
    temp = data[[missing_col, numeric_col]].dropna()
    group_missing = temp.loc[temp[missing_col] == 1, numeric_col]
    group_observed = temp.loc[temp[missing_col] == 0, numeric_col]
    if len(group_missing) < min_group_size or len(group_observed) < min_group_size:
        return None
    try:
        statistic, p_value = stats.mannwhitneyu(group_missing, group_observed, alternative="two-sided")
    except Exception:
        return None
    return {
        "test_type": "Mann-Whitney U",
        "missing_indicator": missing_col,
        "explainer": numeric_col,
        "group_missing_median": group_missing.median(),
        "group_observed_median": group_observed.median(),
        "p_value": p_value,
        "n_missing_group": len(group_missing),
        "n_observed_group": len(group_observed)
    }


def chi_square_missingness_test(data, missing_col, categorical_col, max_categories=40):
    temp = data[[missing_col, categorical_col]].dropna().copy()
    if temp[missing_col].nunique() < 2:
        return None
    if temp[categorical_col].nunique() > max_categories:
        top_categories = temp[categorical_col].value_counts().head(max_categories).index
        temp[categorical_col] = np.where(temp[categorical_col].isin(top_categories), temp[categorical_col], "OTHER")
    contingency = pd.crosstab(temp[missing_col], temp[categorical_col])
    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        return None
    try:
        chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
    except Exception:
        return None
    return {
        "test_type": "Chi-square",
        "missing_indicator": missing_col,
        "explainer": categorical_col,
        "chi2": chi2,
        "degrees_of_freedom": dof,
        "p_value": p_value,
        "n_rows": len(temp),
        "n_categories": temp[categorical_col].nunique()
    }

association_results = []
for missing_indicator in missing_indicator_columns:
    for numeric_col in candidate_numeric_explainers:
        result = mann_whitney_missingness_test(nyc311, missing_indicator, numeric_col)
        if result is not None:
            association_results.append(result)
    for categorical_col in candidate_categorical_explainers:
        result = chi_square_missingness_test(nyc311, missing_indicator, categorical_col)
        if result is not None:
            association_results.append(result)

missingness_association_tests = pd.DataFrame(association_results)
if len(missingness_association_tests) > 0:
    missingness_association_tests = missingness_association_tests.sort_values("p_value", ascending=True).reset_index(drop=True)

display(missingness_association_tests.head(100))
missingness_association_tests.to_csv(OUTPUT_DIR / "nyc311_missingness_association_tests.csv", index=False)


## Section 13 — Evidence-Based Missingness Classification

We classify missingness using statistical evidence and NYC 311 domain reasoning.

Examples:

- `Closed Date` missingness is **MAR** because it strongly depends on request status.
- Coordinate missingness is often **MAR** because it depends on address type, borough, and report channel.
- Some address fields may be **MAR / Suspected MNAR** because users may omit address information intentionally or due to uncertainty.

In [ ]:
# ============================================================
# Section 13: Evidence-Based Missingness Classification
# ============================================================

SIGNIFICANCE_LEVEL = 0.01

classification_rows = []
for col in important_missingness_columns:
    indicator_candidates = [m for m in missing_indicator_columns if m.startswith("is_missing__" + col.replace(" ", "_").replace("(", "").replace(")", ""))]
    indicator = indicator_candidates[0] if indicator_candidates else None
    missing_rate = nyc311[col].isna().mean()

    if indicator is not None and len(missingness_association_tests) > 0:
        sig_count = int(((missingness_association_tests["missing_indicator"] == indicator) & (missingness_association_tests["p_value"] < SIGNIFICANCE_LEVEL)).sum())
        top_explainers = missingness_association_tests[
            (missingness_association_tests["missing_indicator"] == indicator) &
            (missingness_association_tests["p_value"] < SIGNIFICANCE_LEVEL)
        ].head(5)["explainer"].tolist()
    else:
        sig_count = 0
        top_explainers = []

    col_lower = col.lower()
    if missing_rate == 0:
        mechanism = "No missing"
    elif col in [CLOSED_DATE_COL, "Resolution Description", "Resolution Action Updated Date"]:
        mechanism = "MAR / post-outcome missingness"
    elif col in ["Latitude", "Longitude", "Incident Zip", "Incident Address", "Cross Street 1", "Cross Street 2"]:
        mechanism = "MAR / Suspected MNAR"
    elif sig_count >= 3:
        mechanism = "MAR"
    elif missing_rate <= 0.05 and sig_count == 0:
        mechanism = "Approximately MCAR"
    else:
        mechanism = "Unclear / Needs domain review"

    classification_rows.append({
        "column": col,
        "missing_rate": missing_rate,
        "significant_association_count": sig_count,
        "top_associated_observed_features": ", ".join(top_explainers),
        "missingness_mechanism": mechanism
    })

missingness_classification_table = pd.DataFrame(classification_rows).sort_values("missing_rate", ascending=False).reset_index(drop=True)
display(missingness_classification_table)
missingness_classification_table.to_csv(OUTPUT_DIR / "nyc311_missingness_classification.csv", index=False)


## Section 14 — Imputation Strategy Design

Imputation must match the variable type and missingness mechanism.

General rules:

- Numerical variables: median imputation using training data later
- Categorical variables: explicit `Unknown` category
- Missingness indicators: preserved when missingness is informative
- Post-outcome fields: excluded from predictive features to avoid leakage

In [ ]:
# ============================================================
# Section 14: Imputation Strategy Design
# ============================================================

post_outcome_or_leakage_columns = [
    CLOSED_DATE_COL,
    "closed_datetime",
    "Status",
    "Due Date",
    "Resolution Description",
    "Resolution Action Updated Date",
    REGRESSION_TARGET_COL,
    "valid_resolution_time_label",
    "target_missing_closed_date",
    "target_negative_resolution_time",
    "target_extreme_resolution_time_gt_365_days"
]
post_outcome_or_leakage_columns = [col for col in post_outcome_or_leakage_columns if col in nyc311.columns]

identifier_columns = [
    UNIQUE_KEY_COL,
    "near_duplicate_key",
    "Location"
]
identifier_columns = [col for col in identifier_columns if col in nyc311.columns]

raw_datetime_columns = [
    CREATED_DATE_COL,
    CLOSED_DATE_COL,
    "created_datetime",
    "closed_datetime",
    "created_hour_window"
]
raw_datetime_columns = [col for col in raw_datetime_columns if col in nyc311.columns]

imputation_rows = []
for col in nyc311.columns:
    if col in post_outcome_or_leakage_columns:
        strategy = "drop_from_predictive_features"
        reason = "Post-outcome or target-related variable; using it would create leakage."
    elif col in identifier_columns:
        strategy = "drop_identifier"
        reason = "Identifier-like variable that may memorize rows instead of generalizing."
    elif col in raw_datetime_columns:
        strategy = "drop_after_temporal_feature_engineering"
        reason = "Raw datetime is transformed into numerical temporal features."
    elif pd.api.types.is_numeric_dtype(nyc311[col]):
        strategy = "median_imputation_later_in_preprocessor"
        reason = "Median is robust to skewness and outliers."
    else:
        strategy = "Unknown_category_later_in_preprocessor"
        reason = "Explicit Unknown preserves missingness as an informative category."

    imputation_rows.append({
        "column": col,
        "dtype": str(nyc311[col].dtype),
        "missing_rate": nyc311[col].isna().mean(),
        "imputation_or_handling_strategy": strategy,
        "reason": reason
    })

imputation_strategy_table = pd.DataFrame(imputation_rows).sort_values("missing_rate", ascending=False).reset_index(drop=True)
display(imputation_strategy_table)
imputation_strategy_table.to_csv(OUTPUT_DIR / "nyc311_imputation_strategy.csv", index=False)


## Section 15 — Outlier Analysis Overview

The assignment requires comparing Z-score, IQR, and Isolation Forest.

For NYC 311, important outliers include:

- Extremely long resolution times
- Invalid or unusual geographic coordinates
- Large near-duplicate clusters
- Rare temporal/reporting patterns

We compare three methods:

1. Z-score
2. IQR
3. Simple Isolation Forest from scratch using NumPy

In [ ]:
# ============================================================
# Section 15: Numeric Distribution Profile for Outlier Analysis
# ============================================================

# Important leakage rule:
# Resolution time is a target for the regression task. We may summarize it for label-quality
# and target-distribution discussion, but we must not use it to create outlier flags that later
# enter predictive modeling.

target_outlier_profile_columns = [
    REGRESSION_TARGET_COL
]
target_outlier_profile_columns = [
    col for col in target_outlier_profile_columns
    if col in nyc311.columns and pd.api.types.is_numeric_dtype(nyc311[col])
]

# Leakage-safe predictive outlier features: only variables available at request creation time.
outlier_numeric_columns = [
    "Latitude",
    "Longitude",
    "near_duplicate_cluster_size",
    "created_year",
    "created_month",
    "created_dayofweek",
    "created_hour",
    "created_is_weekend",
    "created_is_after_hours",
    "has_geo_coordinates"
]
outlier_numeric_columns = [
    col for col in outlier_numeric_columns
    if col in nyc311.columns and pd.api.types.is_numeric_dtype(nyc311[col])
]

# Profile both target and predictive feature variables, but later outlier flags are built only
# from outlier_numeric_columns.
profile_columns = list(dict.fromkeys(target_outlier_profile_columns + outlier_numeric_columns))

outlier_profile_rows = []
for col in profile_columns:
    x = nyc311[col].dropna()
    if len(x) == 0:
        continue
    outlier_profile_rows.append({
        "column": col,
        "used_for_predictive_outlier_flags": col in outlier_numeric_columns,
        "target_or_feature": "regression_target_profile_only" if col in target_outlier_profile_columns else "predictive_feature",
        "non_missing_count": len(x),
        "missing_rate": nyc311[col].isna().mean(),
        "mean": x.mean(),
        "median": x.median(),
        "std": x.std(),
        "min": x.min(),
        "p01": x.quantile(0.01),
        "p05": x.quantile(0.05),
        "p25": x.quantile(0.25),
        "p75": x.quantile(0.75),
        "p95": x.quantile(0.95),
        "p99": x.quantile(0.99),
        "max": x.max(),
        "skewness": x.skew()
    })

outlier_profile_table = pd.DataFrame(outlier_profile_rows).sort_values("skewness", ascending=False).reset_index(drop=True)
display(outlier_profile_table)
outlier_profile_table.to_csv(OUTPUT_DIR / "nyc311_outlier_numeric_profile.csv", index=False)

outlier_feature_usage_table = pd.DataFrame({
    "column": profile_columns,
    "used_for_predictive_outlier_flags": [col in outlier_numeric_columns for col in profile_columns],
    "reason": [
        "Excluded from predictive outlier flags because it is the regression target."
        if col in target_outlier_profile_columns
        else "Included because it is available at request creation time."
        for col in profile_columns
    ]
})
display(outlier_feature_usage_table)
outlier_feature_usage_table.to_csv(OUTPUT_DIR / "nyc311_outlier_feature_usage_audit.csv", index=False)

for col in [REGRESSION_TARGET_COL, "near_duplicate_cluster_size", "created_hour"]:
    if col in nyc311.columns:
        x = nyc311[col].dropna()
        plt.figure(figsize=(9, 4))
        plt.hist(x, bins=60)
        plt.title(f"Raw Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.grid(alpha=0.3)
        plt.show()
        if len(x) > 0 and x.min() >= 0:
            plt.figure(figsize=(9, 4))
            plt.hist(np.log1p(x), bins=60)
            plt.title(f"Log1p Distribution of {col}")
            plt.xlabel(f"log1p({col})")
            plt.ylabel("Frequency")
            plt.grid(alpha=0.3)
            plt.show()

print("Leakage-safe outlier columns used for Z-score, IQR, and Simple Isolation Forest:")
print(outlier_numeric_columns)


## Section 16 — Z-score and IQR Outlier Detection

Z-score is sensitive to mean and standard deviation.

IQR is more robust for skewed data, but it may over-flag naturally long-tailed resolution times.

In [ ]:
# ============================================================
# Section 16: Z-score and IQR Outlier Detection
# ============================================================


def detect_zscore_outliers(data, columns, threshold=3.0):
    flags = pd.DataFrame(index=data.index)
    summary_rows = []
    for col in columns:
        x = pd.to_numeric(data[col], errors="coerce")
        mean_value = x.mean(skipna=True)
        std_value = x.std(skipna=True)
        if pd.isna(std_value) or std_value == 0:
            flags[f"z_outlier__{col}"] = False
            continue
        z = ((x - mean_value) / std_value).abs()
        flag = (z > threshold).fillna(False)
        flags[f"z_outlier__{col}"] = flag
        summary_rows.append({
            "column": col,
            "method": "Z-score",
            "threshold": threshold,
            "mean": mean_value,
            "std": std_value,
            "outlier_count": int(flag.sum()),
            "outlier_rate": flag.mean()
        })
    flags["zscore_outlier_any"] = flags.any(axis=1)
    return flags, pd.DataFrame(summary_rows)


def detect_iqr_outliers(data, columns, multiplier=1.5):
    flags = pd.DataFrame(index=data.index)
    summary_rows = []
    for col in columns:
        x = pd.to_numeric(data[col], errors="coerce")
        q1 = x.quantile(0.25)
        q3 = x.quantile(0.75)
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            flags[f"iqr_outlier__{col}"] = False
            continue
        lower = q1 - multiplier * iqr
        upper = q3 + multiplier * iqr
        flag = ((x < lower) | (x > upper)).fillna(False)
        flags[f"iqr_outlier__{col}"] = flag
        summary_rows.append({
            "column": col,
            "method": "IQR",
            "multiplier": multiplier,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_count": int(flag.sum()),
            "outlier_rate": flag.mean()
        })
    flags["iqr_outlier_any"] = flags.any(axis=1)
    return flags, pd.DataFrame(summary_rows)

zscore_flags, zscore_summary = detect_zscore_outliers(nyc311, outlier_numeric_columns, threshold=3.0)
iqr_flags, iqr_summary = detect_iqr_outliers(nyc311, outlier_numeric_columns, multiplier=1.5)

zscore_summary = zscore_summary.sort_values("outlier_rate", ascending=False).reset_index(drop=True)
iqr_summary = iqr_summary.sort_values("outlier_rate", ascending=False).reset_index(drop=True)

display(zscore_summary)
display(iqr_summary)

zscore_summary.to_csv(OUTPUT_DIR / "nyc311_zscore_outlier_summary.csv", index=False)
iqr_summary.to_csv(OUTPUT_DIR / "nyc311_iqr_outlier_summary.csv", index=False)


## Section 17 — Simple Isolation Forest from Scratch

Isolation Forest is multivariate. It can detect unusual combinations of features, not just extreme values in one column.

This implementation uses only NumPy and custom tree logic.

In [ ]:
# ============================================================
# Section 17: Simple Isolation Forest from Scratch
# ============================================================


def average_path_length(n):
    n = np.asarray(n, dtype=float)
    result = np.zeros_like(n, dtype=float)
    mask_gt_2 = n > 2
    mask_eq_2 = n == 2
    result[mask_eq_2] = 1.0
    result[mask_gt_2] = 2.0 * (np.log(n[mask_gt_2] - 1.0) + 0.5772156649) - (2.0 * (n[mask_gt_2] - 1.0) / n[mask_gt_2])
    return result

class IsolationTreeNode:
    def __init__(self, size, depth, feature_index=None, split_value=None, left=None, right=None):
        self.size = size
        self.depth = depth
        self.feature_index = feature_index
        self.split_value = split_value
        self.left = left
        self.right = right
    @property
    def is_external(self):
        return self.left is None and self.right is None

class IsolationTree:
    def __init__(self, max_depth, random_state=None):
        self.max_depth = max_depth
        self.rng = np.random.default_rng(random_state)
        self.root = None
    def fit(self, X):
        self.root = self._build_tree(np.asarray(X, dtype=float), depth=0)
        return self
    def _build_tree(self, X, depth):
        n_samples, n_features = X.shape
        if depth >= self.max_depth or n_samples <= 1:
            return IsolationTreeNode(size=n_samples, depth=depth)
        mins = np.nanmin(X, axis=0)
        maxs = np.nanmax(X, axis=0)
        valid_features = np.where(maxs > mins)[0]
        if len(valid_features) == 0:
            return IsolationTreeNode(size=n_samples, depth=depth)
        feature_index = int(self.rng.choice(valid_features))
        split_value = float(self.rng.uniform(mins[feature_index], maxs[feature_index]))
        left_mask = X[:, feature_index] < split_value
        right_mask = ~left_mask
        if left_mask.sum() == 0 or right_mask.sum() == 0:
            return IsolationTreeNode(size=n_samples, depth=depth)
        return IsolationTreeNode(
            size=n_samples,
            depth=depth,
            feature_index=feature_index,
            split_value=split_value,
            left=self._build_tree(X[left_mask], depth + 1),
            right=self._build_tree(X[right_mask], depth + 1)
        )
    def path_length_one(self, x, node=None, depth=0):
        if node is None:
            node = self.root
        if node.is_external:
            return depth + float(average_path_length(np.array([node.size]))[0])
        if x[node.feature_index] < node.split_value:
            return self.path_length_one(x, node.left, depth + 1)
        return self.path_length_one(x, node.right, depth + 1)
    def path_length(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self.path_length_one(row) for row in X], dtype=float)

class SimpleIsolationForest:
    def __init__(self, n_trees=100, sample_size=256, max_depth=None, random_state=42):
        self.n_trees = n_trees
        self.sample_size = sample_size
        self.max_depth = max_depth
        self.random_state = random_state
        self.rng = np.random.default_rng(random_state)
        self.trees = []
        self.actual_sample_size_ = None
    def fit(self, X):
        X = np.asarray(X, dtype=float)
        n_samples = X.shape[0]
        self.actual_sample_size_ = min(self.sample_size, n_samples)
        if self.max_depth is None:
            self.max_depth = int(np.ceil(np.log2(self.actual_sample_size_)))
        self.trees = []
        for _ in range(self.n_trees):
            idx = self.rng.choice(n_samples, size=self.actual_sample_size_, replace=False)
            tree = IsolationTree(max_depth=self.max_depth, random_state=self.rng.integers(0, 10**9))
            tree.fit(X[idx])
            self.trees.append(tree)
        return self
    def path_length(self, X):
        X = np.asarray(X, dtype=float)
        lengths = np.zeros((X.shape[0], len(self.trees)), dtype=float)
        for j, tree in enumerate(self.trees):
            lengths[:, j] = tree.path_length(X)
        return lengths.mean(axis=1)
    def anomaly_score(self, X):
        avg_lengths = self.path_length(X)
        c_n = float(average_path_length(np.array([self.actual_sample_size_]))[0])
        if c_n == 0:
            return np.ones_like(avg_lengths)
        return 2.0 ** (-avg_lengths / c_n)
    def predict(self, X, contamination=0.05):
        scores = self.anomaly_score(X)
        threshold = np.quantile(scores, 1.0 - contamination)
        labels = (scores >= threshold).astype(int)
        return labels, scores, threshold


def prepare_isolation_matrix(data, columns, clip_value=10.0):
    X_df = data[columns].copy()
    transform_rows = []
    for col in X_df.columns:
        x = pd.to_numeric(X_df[col], errors="coerce")
        skew = x.dropna().skew()
        log_applied = False
        if pd.notna(skew) and skew > 1.0 and x.min(skipna=True) >= 0:
            x = np.log1p(x)
            log_applied = True
        med = x.median()
        if pd.isna(med): med = 0.0
        x = x.fillna(med)
        q1, q3 = x.quantile(0.25), x.quantile(0.75)
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0: iqr = 1.0
        X_df[col] = ((x - med) / iqr).clip(-clip_value, clip_value)
        transform_rows.append({"column": col, "original_skewness": skew, "log1p_applied": log_applied, "median": med, "iqr": iqr})
    return X_df.values.astype(float), pd.DataFrame(transform_rows), X_df

X_iso, isolation_transform_table, X_iso_df = prepare_isolation_matrix(nyc311, outlier_numeric_columns)

start_time = time.time()
iso = SimpleIsolationForest(n_trees=100, sample_size=256, random_state=RANDOM_STATE)
iso.fit(X_iso)
iso_labels, iso_scores, iso_threshold = iso.predict(X_iso, contamination=0.05)
iso_runtime = time.time() - start_time

nyc311["simple_iforest_score"] = iso_scores
nyc311["simple_iforest_outlier"] = iso_labels

print("Isolation Forest runtime:", iso_runtime)
print("Threshold:", iso_threshold)
print("Outliers:", nyc311["simple_iforest_outlier"].sum())

display(isolation_transform_table)
isolation_transform_table.to_csv(OUTPUT_DIR / "nyc311_isolation_forest_transform_table.csv", index=False)


## Section 18 — Compare Outlier Detection Methods

Rows detected by multiple methods are more suspicious than rows detected by only one method.

We create a high-confidence outlier indicator based on method agreement.

In [ ]:
# ============================================================
# Section 18: Compare Outlier Detection Methods
# ============================================================

outlier_comparison = pd.DataFrame(index=nyc311.index)
outlier_comparison["zscore_outlier"] = zscore_flags["zscore_outlier_any"].astype(int)
outlier_comparison["iqr_outlier"] = iqr_flags["iqr_outlier_any"].astype(int)
outlier_comparison["simple_iforest_outlier"] = nyc311["simple_iforest_outlier"].astype(int)
outlier_comparison["number_of_methods_flagged"] = outlier_comparison.sum(axis=1)

nyc311["zscore_outlier_any"] = outlier_comparison["zscore_outlier"]
nyc311["iqr_outlier_any"] = outlier_comparison["iqr_outlier"]
nyc311["number_of_outlier_methods_flagged"] = outlier_comparison["number_of_methods_flagged"]
nyc311["high_confidence_outlier"] = (
    (nyc311["simple_iforest_outlier"] == 1) &
    ((nyc311["zscore_outlier_any"] == 1) | (nyc311["iqr_outlier_any"] == 1))
).astype(int)

outlier_method_summary = pd.DataFrame({
    "method": ["Z-score", "IQR", "Simple Isolation Forest", "High-confidence combined"],
    "outlier_count": [
        int(nyc311["zscore_outlier_any"].sum()),
        int(nyc311["iqr_outlier_any"].sum()),
        int(nyc311["simple_iforest_outlier"].sum()),
        int(nyc311["high_confidence_outlier"].sum())
    ],
    "outlier_rate": [
        nyc311["zscore_outlier_any"].mean(),
        nyc311["iqr_outlier_any"].mean(),
        nyc311["simple_iforest_outlier"].mean(),
        nyc311["high_confidence_outlier"].mean()
    ]
})

display(outlier_method_summary)
display(outlier_comparison["number_of_methods_flagged"].value_counts().sort_index())

inspection_cols = [col for col in [UNIQUE_KEY_COL, CREATED_DATE_COL, CLOSED_DATE_COL, CLASSIFICATION_TARGET_COL, REGRESSION_TARGET_COL, "Borough", "Incident Zip", "Latitude", "Longitude", "near_duplicate_cluster_size", "simple_iforest_score", "number_of_outlier_methods_flagged"] if col in nyc311.columns]
display(nyc311.sort_values(["number_of_outlier_methods_flagged", "simple_iforest_score"], ascending=[False, False])[inspection_cols].head(30))

outlier_method_summary.to_csv(OUTPUT_DIR / "nyc311_outlier_method_summary.csv", index=False)
nyc311.to_csv(OUTPUT_DIR / "nyc311_after_outlier_analysis.csv", index=False)


## Section 19 — Feature Quality Analysis Overview

Before modeling, we check:

- Identifier features
- Post-outcome leakage
- Target leakage
- High-cardinality categorical features
- Near-constant features
- Redundant numerical features

The feature strategy is task-specific:

- Complaint category classification must not use the complaint category itself or fields that trivially encode it.
- Resolution time regression must not use `Closed Date` or any post-resolution fields.

In [ ]:
# ============================================================
# Section 19: Feature Inventory
# ============================================================


def is_boolean_like(series):
    unique_values = set(series.dropna().astype(str).str.lower().unique())
    boolean_sets = [{"0", "1"}, {"true", "false"}, {"t", "f"}, {"yes", "no"}]
    return any(unique_values.issubset(s) for s in boolean_sets) and len(unique_values) <= 2

identifier_keywords = ["key", "id", "address", "street", "location", "bbl", "coordinate", "landmark", "facility", "vehicle", "taxi", "bridge", "road", "ramp"]
post_outcome_keywords = ["closed", "resolution", "status", "due"]

feature_inventory_rows = []
for col in nyc311.columns:
    dtype = str(nyc311[col].dtype)
    unique_count = nyc311[col].nunique(dropna=False)
    unique_rate = unique_count / len(nyc311)
    missing_rate = nyc311[col].isna().mean()
    col_lower = col.lower()

    if col in [CLASSIFICATION_TARGET_COL, REGRESSION_TARGET_COL]:
        ftype = "target"
    elif any(k in col_lower for k in post_outcome_keywords):
        ftype = "post_outcome_or_leakage_risk"
    elif any(k in col_lower for k in identifier_keywords):
        ftype = "identifier_or_high_specificity"
    elif is_boolean_like(nyc311[col]):
        ftype = "boolean_like"
    elif pd.api.types.is_numeric_dtype(nyc311[col]):
        ftype = "numeric"
    elif nyc311[col].dtype == "object":
        ftype = "categorical_or_text"
    else:
        ftype = "other"

    feature_inventory_rows.append({
        "column": col,
        "dtype": dtype,
        "feature_type": ftype,
        "missing_rate": missing_rate,
        "unique_count": unique_count,
        "unique_rate": unique_rate
    })

feature_inventory_table = pd.DataFrame(feature_inventory_rows).sort_values(["feature_type", "missing_rate"], ascending=[True, False]).reset_index(drop=True)
display(feature_inventory_table)
display(feature_inventory_table["feature_type"].value_counts())
feature_inventory_table.to_csv(OUTPUT_DIR / "nyc311_feature_inventory.csv", index=False)


## Section 20 — Near-Constant and High-Cardinality Features

Near-constant features provide little signal.

High-cardinality features can cause sparse matrices and overfitting. Some are meaningful, such as ZIP code; others are row-specific, such as exact addresses.

In [ ]:
# ============================================================
# Section 20: Near-Constant and High-Cardinality Features
# ============================================================

near_constant_rows = []
for col in nyc311.columns:
    vc = nyc311[col].value_counts(dropna=False, normalize=True)
    if len(vc) == 0:
        continue
    dominant_rate = vc.iloc[0]
    unique_count = nyc311[col].nunique(dropna=False)
    if unique_count == 1:
        status = "constant"
    elif dominant_rate >= 0.99:
        status = "near_constant"
    else:
        status = "variable"
    if status != "variable":
        near_constant_rows.append({
            "column": col,
            "status": status,
            "dominant_value": vc.index[0],
            "dominant_rate": dominant_rate,
            "unique_count": unique_count
        })

near_constant_table = pd.DataFrame(near_constant_rows)
if len(near_constant_table) > 0:
    near_constant_table = near_constant_table.sort_values(["status", "dominant_rate"], ascending=[True, False]).reset_index(drop=True)
display(near_constant_table)

categorical_columns = [col for col in nyc311.columns if nyc311[col].dtype == "object"]
high_cardinality_rows = []
for col in categorical_columns:
    unique_count = nyc311[col].nunique(dropna=False)
    unique_rate = unique_count / len(nyc311)
    if unique_count >= 50 or unique_rate >= 0.20:
        col_lower = col.lower()
        if any(k in col_lower for k in ["key", "address", "street", "location", "description"]):
            recommendation = "drop_or_engineer_indicator_only"
        elif any(k in col_lower for k in ["zip", "borough", "city", "community board", "agency", "problem"]):
            recommendation = "keep_with_top_k_or_frequency_encoding"
        else:
            recommendation = "review_later"
        high_cardinality_rows.append({
            "column": col,
            "unique_count": unique_count,
            "unique_rate": unique_rate,
            "top_category_rate": nyc311[col].value_counts(dropna=False, normalize=True).iloc[0],
            "recommendation": recommendation
        })

high_cardinality_table = pd.DataFrame(high_cardinality_rows)
if len(high_cardinality_table) > 0:
    high_cardinality_table = high_cardinality_table.sort_values("unique_count", ascending=False).reset_index(drop=True)
display(high_cardinality_table)

near_constant_table.to_csv(OUTPUT_DIR / "nyc311_near_constant_features.csv", index=False)
high_cardinality_table.to_csv(OUTPUT_DIR / "nyc311_high_cardinality_features.csv", index=False)


## Section 21 — Redundancy and Leakage Analysis

We analyze numerical redundancy using Spearman correlation.

We also create leakage rules for both tasks.

In [ ]:
# ============================================================
# Section 21: Redundancy and Leakage Analysis
# ============================================================

numeric_columns = [col for col in nyc311.columns if pd.api.types.is_numeric_dtype(nyc311[col])]
numeric_corr_columns = [col for col in numeric_columns if nyc311[col].nunique(dropna=True) > 1]

corr_matrix = nyc311[numeric_corr_columns].corr(method="spearman").abs()
redundant_pairs = []
for i, col1 in enumerate(corr_matrix.columns):
    for j, col2 in enumerate(corr_matrix.columns):
        if j <= i:
            continue
        corr_value = corr_matrix.loc[col1, col2]
        if pd.notna(corr_value) and corr_value >= 0.90:
            redundant_pairs.append({"feature_1": col1, "feature_2": col2, "abs_spearman_corr": corr_value})

redundant_pairs_table = pd.DataFrame(redundant_pairs)
if len(redundant_pairs_table) > 0:
    redundant_pairs_table = redundant_pairs_table.sort_values("abs_spearman_corr", ascending=False).reset_index(drop=True)
display(redundant_pairs_table.head(100))


def complaint_classification_leakage_risk(col):
    col_lower = col.lower()
    if col == CLASSIFICATION_TARGET_COL:
        return "direct_target"
    if "problem detail" in col_lower or "descriptor" in col_lower:
        return "high_risk_label_hierarchy"
    if "agency" in col_lower:
        return "high_risk_operational_assignment"
    if any(k in col_lower for k in ["resolution", "closed", "status", "due"]):
        return "post_outcome_leakage"
    return "no_direct_leakage_detected"


def resolution_regression_leakage_risk(col):
    col_lower = col.lower()
    if col == REGRESSION_TARGET_COL:
        return "direct_target"
    if any(k in col_lower for k in ["closed", "resolution", "status", "due", "valid_resolution", "target_"]):
        return "post_outcome_or_target_derived"
    return "no_direct_leakage_detected"

leakage_rows = []
for col in nyc311.columns:
    c_risk = complaint_classification_leakage_risk(col)
    r_risk = resolution_regression_leakage_risk(col)
    if c_risk != "no_direct_leakage_detected" or r_risk != "no_direct_leakage_detected":
        leakage_rows.append({
            "column": col,
            "complaint_classification_risk": c_risk,
            "resolution_regression_risk": r_risk
        })

leakage_candidate_table = pd.DataFrame(leakage_rows)
display(leakage_candidate_table)

redundant_pairs_table.to_csv(OUTPUT_DIR / "nyc311_redundant_numeric_pairs.csv", index=False)
leakage_candidate_table.to_csv(OUTPUT_DIR / "nyc311_leakage_candidates.csv", index=False)


## Section 22 — Final Feature Decision Table

We combine all feature-quality signals into one table and create two task-specific feature lists.

In [ ]:
# ============================================================
# Section 22: Final Feature Decision Table
# ============================================================

near_constant_set = set(near_constant_table["column"]) if len(near_constant_table) > 0 else set()
high_cardinality_map = dict(zip(high_cardinality_table["column"], high_cardinality_table["recommendation"])) if len(high_cardinality_table) > 0 else {}
redundant_set = set()
if len(redundant_pairs_table) > 0:
    redundant_set.update(redundant_pairs_table["feature_1"])
    redundant_set.update(redundant_pairs_table["feature_2"])


def quality_flag(col):
    if col in near_constant_set:
        return "near_constant_or_constant"
    if col in high_cardinality_map:
        return high_cardinality_map[col]
    if col in redundant_set:
        return "redundancy_review"
    return "no_major_quality_issue_detected"


def complaint_recommendation(col):
    leak = complaint_classification_leakage_risk(col)
    if leak != "no_direct_leakage_detected":
        return "drop_for_complaint_classification_leakage_risk"
    if resolution_regression_leakage_risk(col) != "no_direct_leakage_detected":
        return "drop_post_outcome_feature"
    q = quality_flag(col)
    if q == "near_constant_or_constant":
        return "drop_near_constant"
    if q == "drop_or_engineer_indicator_only":
        return "drop_high_specificity_text_or_identifier"
    return "keep_candidate"


def resolution_recommendation(col):
    leak = resolution_regression_leakage_risk(col)
    if leak != "no_direct_leakage_detected":
        return "drop_for_resolution_regression_leakage_risk"
    if col == CLASSIFICATION_TARGET_COL:
        return "keep_candidate"  # complaint type is known at request creation and useful for resolution time.
    q = quality_flag(col)
    if q == "near_constant_or_constant":
        return "drop_near_constant"
    if q == "drop_or_engineer_indicator_only":
        return "drop_high_specificity_text_or_identifier"
    return "keep_candidate"

feature_decision_rows = []
for col in nyc311.columns:
    inventory_row = feature_inventory_table[feature_inventory_table["column"] == col].iloc[0]
    feature_decision_rows.append({
        "column": col,
        "dtype": str(nyc311[col].dtype),
        "feature_type": inventory_row["feature_type"],
        "missing_rate": nyc311[col].isna().mean(),
        "unique_count": nyc311[col].nunique(dropna=False),
        "quality_flag": quality_flag(col),
        "complaint_classification_leakage_risk": complaint_classification_leakage_risk(col),
        "resolution_regression_leakage_risk": resolution_regression_leakage_risk(col),
        "complaint_classification_recommendation": complaint_recommendation(col),
        "resolution_regression_recommendation": resolution_recommendation(col)
    })

feature_quality_decision_table = pd.DataFrame(feature_decision_rows)
display(feature_quality_decision_table.sort_values(["complaint_classification_recommendation", "resolution_regression_recommendation", "missing_rate"], ascending=[True, True, False]))

complaint_candidate_features = feature_quality_decision_table.loc[
    feature_quality_decision_table["complaint_classification_recommendation"] == "keep_candidate", "column"
].tolist()

resolution_candidate_features = feature_quality_decision_table.loc[
    feature_quality_decision_table["resolution_regression_recommendation"] == "keep_candidate", "column"
].tolist()

# Defensive removals
# These variables are either direct targets, row identifiers, post-outcome variables, or
# global outlier-analysis artifacts. They are allowed for data-quality/failure analysis,
# but not as model inputs.
global_outlier_feature_leakage_candidates = [
    "simple_iforest_score",
    "simple_iforest_outlier",
    "zscore_outlier_any",
    "iqr_outlier_any",
    "number_of_outlier_methods_flagged",
    "high_confidence_outlier"
]

post_outcome_model_leakage_candidates = [
    CLASSIFICATION_TARGET_COL,
    REGRESSION_TARGET_COL,
    UNIQUE_KEY_COL,
    CLOSED_DATE_COL,
    "closed_datetime",
    "Status",
    "Due Date",
    "Resolution Description",
    "Resolution Action Updated Date",
    "valid_resolution_time_label",
    "target_missing_created_date",
    "target_missing_closed_date",
    "target_negative_resolution_time",
    "target_extreme_resolution_time_gt_365_days"
]

model_input_forbidden_columns = set(global_outlier_feature_leakage_candidates + post_outcome_model_leakage_candidates)

complaint_candidate_features = [c for c in complaint_candidate_features if c not in model_input_forbidden_columns]
resolution_candidate_features = [c for c in resolution_candidate_features if c not in model_input_forbidden_columns]

model_feature_safety_table = pd.DataFrame({
    "forbidden_model_input_column": sorted(model_input_forbidden_columns),
    "reason": [
        "Direct target, post-outcome variable, row identifier, or global outlier-analysis artifact."
        for _ in sorted(model_input_forbidden_columns)
    ]
})
display(model_feature_safety_table)
model_feature_safety_table.to_csv(OUTPUT_DIR / "nyc311_model_feature_safety_table.csv", index=False)

print("Complaint classification candidate features:", len(complaint_candidate_features))
print("Resolution regression candidate features:", len(resolution_candidate_features))
print("Complaint features preview:", complaint_candidate_features[:80])
print("Resolution features preview:", resolution_candidate_features[:80])

feature_quality_decision_table.to_csv(OUTPUT_DIR / "nyc311_feature_quality_decision_table.csv", index=False)
pd.DataFrame({"feature": complaint_candidate_features}).to_csv(OUTPUT_DIR / "nyc311_complaint_candidate_features.csv", index=False)
pd.DataFrame({"feature": resolution_candidate_features}).to_csv(OUTPUT_DIR / "nyc311_resolution_candidate_features.csv", index=False)


## Section 23 — Modeling Dataset Construction

We now create two modeling datasets:

1. Complaint category classification
2. Resolution time regression

The train/test split and preprocessing are implemented from scratch.

In [ ]:
# ============================================================
# Section 23: Modeling Dataset Construction
# ============================================================

# Classification dataset
classification_valid_mask = nyc311[CLASSIFICATION_TARGET_COL].notna() & nyc311[CLASSIFICATION_TARGET_COL].isin(selected_categories)
df_complaint_task = nyc311.loc[classification_valid_mask].copy().reset_index(drop=True)

class_names = sorted(df_complaint_task[CLASSIFICATION_TARGET_COL].unique())
class_to_index = {c: i for i, c in enumerate(class_names)}
index_to_class = {i: c for c, i in class_to_index.items()}
df_complaint_task["complaint_label"] = df_complaint_task[CLASSIFICATION_TARGET_COL].map(class_to_index).astype(int)

# Regression dataset
regression_valid_mask = nyc311["valid_resolution_time_label"] == 1
df_resolution_task = nyc311.loc[regression_valid_mask].copy().reset_index(drop=True)
df_resolution_task["log1p_resolution_time_target"] = np.log1p(df_resolution_task[REGRESSION_TARGET_COL])

print("Complaint task shape:", df_complaint_task.shape)
print("Resolution task shape:", df_resolution_task.shape)
print("Number of complaint classes:", len(class_names))

display(df_complaint_task[CLASSIFICATION_TARGET_COL].value_counts())
display(df_resolution_task[REGRESSION_TARGET_COL].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))


## Section 24 — Train/Test Split from Scratch

For complaint classification, we use stratified train/test split to preserve class distribution.

For resolution regression, we use random train/test split.

In [ ]:
# ============================================================
# Section 24: Train/Test Split from Scratch
# ============================================================


def train_test_split_numpy_indices(n_samples, test_size=0.25, random_state=42):
    rng = np.random.default_rng(random_state)
    idx = np.arange(n_samples)
    rng.shuffle(idx)
    n_test = int(np.ceil(test_size * n_samples))
    return idx[n_test:], idx[:n_test]


def stratified_train_test_split_numpy_indices(y, test_size=0.25, random_state=42):
    rng = np.random.default_rng(random_state)
    y = np.asarray(y)
    train_idx, test_idx = [], []
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        rng.shuffle(cls_idx)
        n_test = int(np.ceil(test_size * len(cls_idx)))
        test_idx.extend(cls_idx[:n_test])
        train_idx.extend(cls_idx[n_test:])
    train_idx = np.array(train_idx, dtype=int)
    test_idx = np.array(test_idx, dtype=int)
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    return train_idx, test_idx

complaint_train_idx, complaint_test_idx = stratified_train_test_split_numpy_indices(
    df_complaint_task["complaint_label"].values,
    test_size=0.25,
    random_state=RANDOM_STATE
)

resolution_train_idx, resolution_test_idx = train_test_split_numpy_indices(
    len(df_resolution_task),
    test_size=0.25,
    random_state=RANDOM_STATE
)

print("Complaint train/test:", len(complaint_train_idx), len(complaint_test_idx))
print("Resolution train/test:", len(resolution_train_idx), len(resolution_test_idx))

display(pd.Series(df_complaint_task.iloc[complaint_train_idx]["complaint_label"]).value_counts(normalize=True).sort_index())
display(pd.Series(df_complaint_task.iloc[complaint_test_idx]["complaint_label"]).value_counts(normalize=True).sort_index())


## Section 25 — From-Scratch Tabular Preprocessor

The models require numerical matrices.

This preprocessor implements:

- Numerical median imputation
- Robust scaling with median and IQR
- Categorical `Unknown` handling
- Top-K category selection
- Manual one-hot encoding

All parameters are fitted on the training set only.

In [ ]:
# ============================================================
# Section 25: From-Scratch Tabular Preprocessor
# ============================================================

class FromScratchTabularPreprocessor:
    def __init__(self, max_categories_per_feature=25, clip_value=10.0):
        self.max_categories_per_feature = max_categories_per_feature
        self.clip_value = clip_value
        self.numeric_columns_ = []
        self.categorical_columns_ = []
        self.numeric_medians_ = {}
        self.numeric_iqrs_ = {}
        self.categorical_levels_ = {}
        self.feature_names_ = []
    def fit(self, X_df):
        X_df = X_df.copy()
        self.numeric_columns_ = [c for c in X_df.columns if pd.api.types.is_numeric_dtype(X_df[c]) and X_df[c].nunique(dropna=True) > 1]
        self.categorical_columns_ = [c for c in X_df.columns if c not in self.numeric_columns_]
        for col in self.numeric_columns_:
            x = pd.to_numeric(X_df[col], errors="coerce")
            med = x.median()
            q1, q3 = x.quantile(0.25), x.quantile(0.75)
            iqr = q3 - q1
            if pd.isna(med): med = 0.0
            if pd.isna(iqr) or iqr == 0: iqr = 1.0
            self.numeric_medians_[col] = float(med)
            self.numeric_iqrs_[col] = float(iqr)
        for col in self.categorical_columns_:
            s = X_df[col].fillna("Unknown").astype(str).replace({"nan": "Unknown", "None": "Unknown", "": "Unknown"})
            levels = s.value_counts().head(self.max_categories_per_feature).index.tolist()
            if "Other" not in levels:
                levels.append("Other")
            self.categorical_levels_[col] = levels
        self.feature_names_ = []
        for col in self.numeric_columns_:
            self.feature_names_.append(col)
        for col in self.categorical_columns_:
            for level in self.categorical_levels_[col]:
                self.feature_names_.append(f"{col}__{level}")
        return self
    def transform(self, X_df):
        X_df = X_df.copy()
        matrices = []
        if self.numeric_columns_:
            X_num = np.zeros((len(X_df), len(self.numeric_columns_)), dtype=float)
            for j, col in enumerate(self.numeric_columns_):
                x = pd.to_numeric(X_df[col], errors="coerce").fillna(self.numeric_medians_[col])
                x = ((x - self.numeric_medians_[col]) / self.numeric_iqrs_[col]).clip(-self.clip_value, self.clip_value)
                X_num[:, j] = x.values.astype(float)
            matrices.append(X_num)
        cat_mats = []
        for col in self.categorical_columns_:
            s = X_df[col].fillna("Unknown").astype(str).replace({"nan": "Unknown", "None": "Unknown", "": "Unknown"})
            levels = self.categorical_levels_[col]
            level_to_idx = {v: i for i, v in enumerate(levels)}
            s = s.where(s.isin(level_to_idx), "Other")
            X_cat = np.zeros((len(X_df), len(levels)), dtype=float)
            for i, val in enumerate(s.values):
                X_cat[i, level_to_idx.get(val, level_to_idx["Other"])] = 1.0
            cat_mats.append(X_cat)
        if cat_mats:
            matrices.append(np.hstack(cat_mats))
        if not matrices:
            return np.empty((len(X_df), 0), dtype=float)
        return np.hstack(matrices)
    def fit_transform(self, X_df):
        return self.fit(X_df).transform(X_df)

print("Preprocessor defined.")


## Section 26 — Build Final Modeling Matrices

We build final numerical matrices for both tasks and save the feature names.

In [ ]:
# ============================================================
# Section 26: Build Final Modeling Matrices
# ============================================================

X_complaint_df = df_complaint_task[[c for c in complaint_candidate_features if c in df_complaint_task.columns]].copy()
y_complaint = df_complaint_task["complaint_label"].values.astype(int)

X_resolution_df = df_resolution_task[[c for c in resolution_candidate_features if c in df_resolution_task.columns]].copy()
y_resolution_raw = df_resolution_task[REGRESSION_TARGET_COL].values.astype(float)
y_resolution_log = df_resolution_task["log1p_resolution_time_target"].values.astype(float)

X_complaint_train_df = X_complaint_df.iloc[complaint_train_idx].copy()
X_complaint_test_df = X_complaint_df.iloc[complaint_test_idx].copy()
y_train_complaint = y_complaint[complaint_train_idx]
y_test_complaint = y_complaint[complaint_test_idx]

X_resolution_train_df = X_resolution_df.iloc[resolution_train_idx].copy()
X_resolution_test_df = X_resolution_df.iloc[resolution_test_idx].copy()
y_train_resolution_raw = y_resolution_raw[resolution_train_idx]
y_test_resolution_raw = y_resolution_raw[resolution_test_idx]
y_train_resolution_log = y_resolution_log[resolution_train_idx]
y_test_resolution_log = y_resolution_log[resolution_test_idx]

complaint_preprocessor = FromScratchTabularPreprocessor(max_categories_per_feature=25, clip_value=10.0)
resolution_preprocessor = FromScratchTabularPreprocessor(max_categories_per_feature=25, clip_value=10.0)

X_train_complaint = complaint_preprocessor.fit_transform(X_complaint_train_df)
X_test_complaint = complaint_preprocessor.transform(X_complaint_test_df)

X_train_resolution = resolution_preprocessor.fit_transform(X_resolution_train_df)
X_test_resolution = resolution_preprocessor.transform(X_resolution_test_df)

print("Complaint matrices:", X_train_complaint.shape, X_test_complaint.shape, y_train_complaint.shape, y_test_complaint.shape)
print("Resolution matrices:", X_train_resolution.shape, X_test_resolution.shape, y_train_resolution_log.shape, y_test_resolution_log.shape)

assert not np.isnan(X_train_complaint).any()
assert not np.isnan(X_test_complaint).any()
assert not np.isnan(X_train_resolution).any()
assert not np.isnan(X_test_resolution).any()

MODELING_DIR = OUTPUT_DIR / "modeling_arrays"
MODELING_DIR.mkdir(exist_ok=True)

np.save(MODELING_DIR / "X_train_complaint.npy", X_train_complaint)
np.save(MODELING_DIR / "X_test_complaint.npy", X_test_complaint)
np.save(MODELING_DIR / "y_train_complaint.npy", y_train_complaint)
np.save(MODELING_DIR / "y_test_complaint.npy", y_test_complaint)
np.save(MODELING_DIR / "X_train_resolution.npy", X_train_resolution)
np.save(MODELING_DIR / "X_test_resolution.npy", X_test_resolution)
np.save(MODELING_DIR / "y_train_resolution_raw.npy", y_train_resolution_raw)
np.save(MODELING_DIR / "y_test_resolution_raw.npy", y_test_resolution_raw)
np.save(MODELING_DIR / "y_train_resolution_log.npy", y_train_resolution_log)
np.save(MODELING_DIR / "y_test_resolution_log.npy", y_test_resolution_log)

pd.DataFrame({"encoded_feature": complaint_preprocessor.feature_names_}).to_csv(MODELING_DIR / "complaint_encoded_feature_names.csv", index=False)
pd.DataFrame({"encoded_feature": resolution_preprocessor.feature_names_}).to_csv(MODELING_DIR / "resolution_encoded_feature_names.csv", index=False)
pd.DataFrame({"class_index": list(index_to_class.keys()), "complaint_category": list(index_to_class.values())}).to_csv(MODELING_DIR / "complaint_label_mapping.csv", index=False)


## Section 27 — Metrics from Scratch

We implement regression and classification metrics manually.

In [ ]:
# ============================================================
# Section 27: Metrics from Scratch
# ============================================================


def regression_metrics(y_true, y_pred, label="model"):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = y_true - y_pred
    mae = np.mean(np.abs(err))
    mse = np.mean(err ** 2)
    rmse = np.sqrt(mse)
    ss_res = np.sum(err ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    mape = np.mean(np.abs(err) / np.maximum(np.abs(y_true), 1e-8)) * 100
    return {"model": label, "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2, "MAPE_percent": mape}


def classification_metrics(y_true, y_pred, num_classes, label="model"):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    accuracy = np.mean(y_true == y_pred)
    precisions, recalls, f1s = [], [], []
    for c in range(num_classes):
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))
        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
    return {
        "model": label,
        "accuracy": accuracy,
        "macro_precision": float(np.mean(precisions)),
        "macro_recall": float(np.mean(recalls)),
        "macro_f1": float(np.mean(f1s))
    }


def confusion_matrix_from_scratch(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm


## Section 28 — Modeling Subsamples for Expensive Algorithms

Some from-scratch models, especially KNN and kernel methods, become expensive on large samples.

We keep full matrices but create controlled subsamples for expensive models.

In [ ]:
# ============================================================
# Section 28: Modeling Subsamples for Expensive Algorithms
# ============================================================


def subsample_indices(n, max_n, random_state=42):
    rng = np.random.default_rng(random_state)
    if n <= max_n:
        return np.arange(n)
    return rng.choice(n, size=max_n, replace=False)

MAX_LINEAR_TRAIN = 40_000
MAX_KNN_TRAIN = 6_000
MAX_TREE_TRAIN = 30_000
MAX_KERNEL_TRAIN = 2_000
MAX_TEST_EVAL = 15_000

linear_resolution_idx = subsample_indices(len(X_train_resolution), MAX_LINEAR_TRAIN, RANDOM_STATE)
linear_complaint_idx = subsample_indices(len(X_train_complaint), MAX_LINEAR_TRAIN, RANDOM_STATE)
knn_resolution_idx = subsample_indices(len(X_train_resolution), MAX_KNN_TRAIN, RANDOM_STATE)
knn_complaint_idx = subsample_indices(len(X_train_complaint), MAX_KNN_TRAIN, RANDOM_STATE)
tree_resolution_idx = subsample_indices(len(X_train_resolution), MAX_TREE_TRAIN, RANDOM_STATE)
tree_complaint_idx = subsample_indices(len(X_train_complaint), MAX_TREE_TRAIN, RANDOM_STATE)
kernel_resolution_idx = subsample_indices(len(X_train_resolution), MAX_KERNEL_TRAIN, RANDOM_STATE)
kernel_complaint_idx = subsample_indices(len(X_train_complaint), MAX_KERNEL_TRAIN, RANDOM_STATE)
test_resolution_eval_idx = subsample_indices(len(X_test_resolution), MAX_TEST_EVAL, RANDOM_STATE)
test_complaint_eval_idx = subsample_indices(len(X_test_complaint), MAX_TEST_EVAL, RANDOM_STATE)

print("Subsample sizes prepared.")


## Section 29 — Linear Regression from Scratch

Linear regression is used for resolution-time regression.

We train it on log-transformed resolution time because the raw target is heavily skewed.

In [ ]:
# ============================================================
# Section 29: Linear Regression from Scratch
# ============================================================

class LinearRegressionGD:
    def __init__(self, learning_rate=0.03, n_epochs=600, l2=1e-4, verbose=False):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.l2 = l2
        self.verbose = verbose
        self.weights = None
        self.loss_history = []
    def _add_bias(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])
    def fit(self, X, y):
        Xb = self._add_bias(X)
        y = y.reshape(-1, 1)
        self.weights = np.zeros((Xb.shape[1], 1))
        for epoch in range(self.n_epochs):
            pred = Xb @ self.weights
            err = pred - y
            grad = (Xb.T @ err) / len(Xb)
            grad[1:] += self.l2 * self.weights[1:]
            self.weights -= self.learning_rate * grad
            if epoch % 20 == 0 or epoch == self.n_epochs - 1:
                loss = np.mean(err ** 2) + self.l2 * np.sum(self.weights[1:] ** 2)
                self.loss_history.append(loss)
        return self
    def predict(self, X):
        return (self._add_bias(X) @ self.weights).ravel()

start = time.time()
linreg = LinearRegressionGD(learning_rate=0.03, n_epochs=600, l2=1e-4)
linreg.fit(X_train_resolution[linear_resolution_idx], y_train_resolution_log[linear_resolution_idx])
linreg_runtime = time.time() - start

pred_log = linreg.predict(X_test_resolution[test_resolution_eval_idx])
pred_raw = np.expm1(pred_log)
pred_raw = np.maximum(pred_raw, 0)

linear_regression_result = regression_metrics(
    y_test_resolution_raw[test_resolution_eval_idx],
    pred_raw,
    label="Linear Regression GD"
)
linear_regression_result["runtime_seconds"] = linreg_runtime

display(pd.DataFrame([linear_regression_result]))

plt.figure(figsize=(8,4))
plt.plot(linreg.loss_history)
plt.title("Linear Regression Training Loss")
plt.xlabel("Checkpoint")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.show()


## Section 30 — Multiclass Logistic Regression from Scratch

We use softmax regression for multi-class complaint category classification.

In [ ]:
# ============================================================
# Section 30: Multiclass Logistic Regression from Scratch
# ============================================================

class MulticlassLogisticRegressionGD:
    def __init__(self, learning_rate=0.05, n_epochs=400, l2=1e-4, batch_size=512, random_state=42):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.l2 = l2
        self.batch_size = batch_size
        self.random_state = random_state
        self.W = None
        self.loss_history = []
    def _add_bias(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])
    def _softmax(self, Z):
        Z = Z - np.max(Z, axis=1, keepdims=True)
        expZ = np.exp(Z)
        return expZ / np.sum(expZ, axis=1, keepdims=True)
    def fit(self, X, y, num_classes=None):
        rng = np.random.default_rng(self.random_state)
        Xb = self._add_bias(X)
        y = np.asarray(y, dtype=int)
        if num_classes is None:
            num_classes = int(np.max(y)) + 1
        self.W = np.zeros((Xb.shape[1], num_classes))
        Y = np.eye(num_classes)[y]
        n = len(Xb)
        for epoch in range(self.n_epochs):
            idx = rng.permutation(n)
            for start in range(0, n, self.batch_size):
                batch_idx = idx[start:start+self.batch_size]
                xb = Xb[batch_idx]
                yb = Y[batch_idx]
                probs = self._softmax(xb @ self.W)
                grad = xb.T @ (probs - yb) / len(xb)
                grad[1:] += self.l2 * self.W[1:]
                self.W -= self.learning_rate * grad
            if epoch % 20 == 0 or epoch == self.n_epochs - 1:
                probs_full = self._softmax(Xb @ self.W)
                loss = -np.mean(np.sum(Y * np.log(probs_full + 1e-12), axis=1)) + self.l2 * np.sum(self.W[1:] ** 2)
                self.loss_history.append(loss)
        return self
    def predict_proba(self, X):
        return self._softmax(self._add_bias(X) @ self.W)
    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

num_classes = len(class_names)
start = time.time()
logreg = MulticlassLogisticRegressionGD(learning_rate=0.05, n_epochs=400, l2=1e-4, batch_size=512, random_state=RANDOM_STATE)
logreg.fit(X_train_complaint[linear_complaint_idx], y_train_complaint[linear_complaint_idx], num_classes=num_classes)
logreg_runtime = time.time() - start

pred_complaint = logreg.predict(X_test_complaint[test_complaint_eval_idx])
logistic_result = classification_metrics(y_test_complaint[test_complaint_eval_idx], pred_complaint, num_classes, label="Multiclass Logistic Regression GD")
logistic_result["runtime_seconds"] = logreg_runtime

display(pd.DataFrame([logistic_result]))

plt.figure(figsize=(8,4))
plt.plot(logreg.loss_history)
plt.title("Multiclass Logistic Regression Training Loss")
plt.xlabel("Checkpoint")
plt.ylabel("Cross-Entropy Loss")
plt.grid(alpha=0.3)
plt.show()


## Section 31 — KNN from Scratch

KNN is distance-based.

We implement both regression and classification versions.

In [ ]:
# ============================================================
# Section 31: KNN from Scratch
# ============================================================

class KNNFromScratch:
    def __init__(self, k=7, task="classification"):
        self.k = k
        self.task = task
        self.X_train = None
        self.y_train = None
    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self
    def predict(self, X, batch_size=512):
        X = np.asarray(X, dtype=float)
        preds = []
        for start in range(0, len(X), batch_size):
            xb = X[start:start+batch_size]
            dists = np.sqrt(((xb[:, None, :] - self.X_train[None, :, :]) ** 2).sum(axis=2))
            nn_idx = np.argpartition(dists, kth=min(self.k, len(self.X_train)-1), axis=1)[:, :self.k]
            nn_y = self.y_train[nn_idx]
            if self.task == "regression":
                preds.extend(np.mean(nn_y, axis=1))
            else:
                for row in nn_y:
                    counts = np.bincount(row.astype(int), minlength=num_classes)
                    preds.append(np.argmax(counts))
        return np.array(preds)

# KNN Regression on log target
start = time.time()
knn_reg = KNNFromScratch(k=7, task="regression")
knn_reg.fit(X_train_resolution[knn_resolution_idx], y_train_resolution_log[knn_resolution_idx])
knn_pred_log = knn_reg.predict(X_test_resolution[test_resolution_eval_idx], batch_size=256)
knn_pred_raw = np.maximum(np.expm1(knn_pred_log), 0)
knn_reg_runtime = time.time() - start

knn_reg_result = regression_metrics(y_test_resolution_raw[test_resolution_eval_idx], knn_pred_raw, label="KNN Regression")
knn_reg_result["runtime_seconds"] = knn_reg_runtime

# KNN Classification
start = time.time()
knn_clf = KNNFromScratch(k=7, task="classification")
knn_clf.fit(X_train_complaint[knn_complaint_idx], y_train_complaint[knn_complaint_idx])
knn_pred_class = knn_clf.predict(X_test_complaint[test_complaint_eval_idx], batch_size=256)
knn_clf_runtime = time.time() - start

knn_clf_result = classification_metrics(y_test_complaint[test_complaint_eval_idx], knn_pred_class, num_classes, label="KNN Classification")
knn_clf_result["runtime_seconds"] = knn_clf_runtime

display(pd.DataFrame([knn_reg_result]))
display(pd.DataFrame([knn_clf_result]))


## Section 32 — Decision Tree from Scratch

We implement simple CART-style decision trees for both regression and classification.

In [ ]:
# ============================================================
# Section 32: Decision Tree from Scratch
# ============================================================

class DecisionTreeNode:
    def __init__(self, prediction=None, feature_index=None, threshold=None, left=None, right=None):
        self.prediction = prediction
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
    @property
    def is_leaf(self):
        return self.left is None and self.right is None

class DecisionTreeFromScratch:
    def __init__(self, task="classification", max_depth=8, min_samples_split=50, max_features=30, random_state=42):
        self.task = task
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.rng = np.random.default_rng(random_state)
        self.root = None
    def _prediction(self, y):
        if self.task == "regression":
            return float(np.mean(y))
        counts = np.bincount(y.astype(int), minlength=num_classes)
        return int(np.argmax(counts))
    def _impurity(self, y):
        if self.task == "regression":
            return np.var(y)
        counts = np.bincount(y.astype(int), minlength=num_classes)
        probs = counts / max(counts.sum(), 1)
        return 1.0 - np.sum(probs ** 2)
    def _best_split(self, X, y):
        n, p = X.shape
        parent_impurity = self._impurity(y)
        best_gain, best_feature, best_threshold = 0.0, None, None
        features = np.arange(p)
        if self.max_features is not None and p > self.max_features:
            features = self.rng.choice(p, size=self.max_features, replace=False)
        for j in features:
            values = X[:, j]
            if np.unique(values).size <= 1:
                continue
            thresholds = np.quantile(values, np.linspace(0.1, 0.9, 9))
            thresholds = np.unique(thresholds)
            for thr in thresholds:
                left = values <= thr
                right = ~left
                if left.sum() < self.min_samples_split // 2 or right.sum() < self.min_samples_split // 2:
                    continue
                weighted_impurity = (left.mean() * self._impurity(y[left])) + (right.mean() * self._impurity(y[right]))
                gain = parent_impurity - weighted_impurity
                if gain > best_gain:
                    best_gain, best_feature, best_threshold = gain, j, thr
        return best_feature, best_threshold, best_gain
    def _build(self, X, y, depth):
        pred = self._prediction(y)
        if depth >= self.max_depth or len(y) < self.min_samples_split or self._impurity(y) == 0:
            return DecisionTreeNode(prediction=pred)
        feat, thr, gain = self._best_split(X, y)
        if feat is None or gain <= 1e-12:
            return DecisionTreeNode(prediction=pred)
        left_mask = X[:, feat] <= thr
        right_mask = ~left_mask
        return DecisionTreeNode(
            prediction=pred,
            feature_index=feat,
            threshold=thr,
            left=self._build(X[left_mask], y[left_mask], depth + 1),
            right=self._build(X[right_mask], y[right_mask], depth + 1)
        )
    def fit(self, X, y):
        self.root = self._build(np.asarray(X, dtype=float), np.asarray(y), depth=0)
        return self
    def _predict_one(self, x, node):
        while not node.is_leaf:
            if x[node.feature_index] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.prediction
    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self._predict_one(row, self.root) for row in X])

start = time.time()
tree_reg = DecisionTreeFromScratch(task="regression", max_depth=8, min_samples_split=80, max_features=40, random_state=RANDOM_STATE)
tree_reg.fit(X_train_resolution[tree_resolution_idx], y_train_resolution_log[tree_resolution_idx])
tree_pred_log = tree_reg.predict(X_test_resolution[test_resolution_eval_idx])
tree_pred_raw = np.maximum(np.expm1(tree_pred_log), 0)
tree_reg_runtime = time.time() - start
tree_reg_result = regression_metrics(y_test_resolution_raw[test_resolution_eval_idx], tree_pred_raw, label="Decision Tree Regression")
tree_reg_result["runtime_seconds"] = tree_reg_runtime

start = time.time()
tree_clf = DecisionTreeFromScratch(task="classification", max_depth=8, min_samples_split=80, max_features=40, random_state=RANDOM_STATE)
tree_clf.fit(X_train_complaint[tree_complaint_idx], y_train_complaint[tree_complaint_idx])
tree_pred_class = tree_clf.predict(X_test_complaint[test_complaint_eval_idx]).astype(int)
tree_clf_runtime = time.time() - start
tree_clf_result = classification_metrics(y_test_complaint[test_complaint_eval_idx], tree_pred_class, num_classes, label="Decision Tree Classification")
tree_clf_result["runtime_seconds"] = tree_clf_runtime

display(pd.DataFrame([tree_reg_result]))
display(pd.DataFrame([tree_clf_result]))


## Section 33 — Kernel Functions and Kernel Ridge Regression

Kernel methods are computationally expensive because they require kernel matrices.

We test linear, polynomial, and RBF kernels on controlled subsets.

In [ ]:
# ============================================================
# Section 33: Kernel Functions and KRR from Scratch
# ============================================================


def linear_kernel(X, Z):
    return X @ Z.T


def polynomial_kernel(X, Z, degree=2, coef0=1.0, gamma=None):
    if gamma is None:
        gamma = 1.0 / X.shape[1]
    return (gamma * (X @ Z.T) + coef0) ** degree


def rbf_kernel(X, Z, gamma=None):
    if gamma is None:
        gamma = 1.0 / X.shape[1]
    X_norm = np.sum(X ** 2, axis=1)[:, None]
    Z_norm = np.sum(Z ** 2, axis=1)[None, :]
    sqdist = X_norm + Z_norm - 2 * X @ Z.T
    sqdist = np.maximum(sqdist, 0)
    return np.exp(-gamma * sqdist)


def compute_kernel(X, Z, kernel="rbf", gamma=None, degree=2, coef0=1.0):
    if kernel == "linear":
        return linear_kernel(X, Z)
    if kernel == "poly":
        return polynomial_kernel(X, Z, degree=degree, coef0=coef0, gamma=gamma)
    if kernel == "rbf":
        return rbf_kernel(X, Z, gamma=gamma)
    raise ValueError("Unknown kernel")

class KernelRidgeRegression:
    def __init__(self, kernel="rbf", alpha=1.0, gamma=None, degree=2, coef0=1.0):
        self.kernel = kernel
        self.alpha = alpha
        self.gamma = gamma
        self.degree = degree
        self.coef0 = coef0
        self.X_train = None
        self.dual_coef = None
    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        K = compute_kernel(self.X_train, self.X_train, self.kernel, self.gamma, self.degree, self.coef0)
        self.dual_coef = np.linalg.solve(K + self.alpha * np.eye(K.shape[0]), y)
        return self
    def predict(self, X):
        K = compute_kernel(np.asarray(X, dtype=float), self.X_train, self.kernel, self.gamma, self.degree, self.coef0)
        return K @ self.dual_coef

kernel_regression_results = []
for kernel_name in ["linear", "poly", "rbf"]:
    start = time.time()
    krr = KernelRidgeRegression(kernel=kernel_name, alpha=1.0, gamma=1.0 / X_train_resolution.shape[1], degree=2)
    krr.fit(X_train_resolution[kernel_resolution_idx], y_train_resolution_log[kernel_resolution_idx])
    pred_log = krr.predict(X_test_resolution[test_resolution_eval_idx])
    pred_raw = np.maximum(np.expm1(pred_log), 0)
    runtime = time.time() - start
    result = regression_metrics(y_test_resolution_raw[test_resolution_eval_idx], pred_raw, label=f"KRR {kernel_name}")
    result["runtime_seconds"] = runtime
    result["kernel"] = kernel_name
    kernel_regression_results.append(result)

kernel_regression_results_df = pd.DataFrame(kernel_regression_results)
display(kernel_regression_results_df)


## Section 34 — Kernel KNN from Scratch

Kernel KNN uses kernel-induced distance:

`d²(x,z) = K(x,x) + K(z,z) - 2K(x,z)`

In [ ]:
# ============================================================
# Section 34: Kernel KNN from Scratch
# ============================================================

class KernelKNNFromScratch:
    def __init__(self, k=7, task="classification", kernel="rbf", gamma=None, degree=2, coef0=1.0):
        self.k = k
        self.task = task
        self.kernel = kernel
        self.gamma = gamma
        self.degree = degree
        self.coef0 = coef0
    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        self.K_train_diag = np.diag(compute_kernel(self.X_train, self.X_train, self.kernel, self.gamma, self.degree, self.coef0))
        return self
    def predict(self, X, batch_size=256):
        X = np.asarray(X, dtype=float)
        preds = []
        for start in range(0, len(X), batch_size):
            xb = X[start:start+batch_size]
            K_xt = compute_kernel(xb, self.X_train, self.kernel, self.gamma, self.degree, self.coef0)
            K_xx = np.diag(compute_kernel(xb, xb, self.kernel, self.gamma, self.degree, self.coef0))[:, None]
            d2 = K_xx + self.K_train_diag[None, :] - 2 * K_xt
            d2 = np.maximum(d2, 0)
            nn_idx = np.argpartition(d2, kth=min(self.k, len(self.X_train)-1), axis=1)[:, :self.k]
            nn_y = self.y_train[nn_idx]
            if self.task == "regression":
                preds.extend(np.mean(nn_y, axis=1))
            else:
                for row in nn_y:
                    counts = np.bincount(row.astype(int), minlength=num_classes)
                    preds.append(np.argmax(counts))
        return np.array(preds)

kernel_knn_results_reg = []
for kernel_name in ["linear", "poly", "rbf"]:
    start = time.time()
    model = KernelKNNFromScratch(k=7, task="regression", kernel=kernel_name, gamma=1.0 / X_train_resolution.shape[1], degree=2)
    model.fit(X_train_resolution[kernel_resolution_idx], y_train_resolution_log[kernel_resolution_idx])
    pred_log = model.predict(X_test_resolution[test_resolution_eval_idx], batch_size=128)
    pred_raw = np.maximum(np.expm1(pred_log), 0)
    runtime = time.time() - start
    result = regression_metrics(y_test_resolution_raw[test_resolution_eval_idx], pred_raw, label=f"Kernel KNN Regression {kernel_name}")
    result["runtime_seconds"] = runtime
    result["kernel"] = kernel_name
    kernel_knn_results_reg.append(result)

display(pd.DataFrame(kernel_knn_results_reg))

kernel_knn_results_clf = []
for kernel_name in ["linear", "poly", "rbf"]:
    start = time.time()
    model = KernelKNNFromScratch(k=7, task="classification", kernel=kernel_name, gamma=1.0 / X_train_complaint.shape[1], degree=2)
    model.fit(X_train_complaint[kernel_complaint_idx], y_train_complaint[kernel_complaint_idx])
    pred = model.predict(X_test_complaint[test_complaint_eval_idx], batch_size=128)
    runtime = time.time() - start
    result = classification_metrics(y_test_complaint[test_complaint_eval_idx], pred, num_classes, label=f"Kernel KNN Classification {kernel_name}")
    result["runtime_seconds"] = runtime
    result["kernel"] = kernel_name
    kernel_knn_results_clf.append(result)

display(pd.DataFrame(kernel_knn_results_clf))


## Section 35 — Simplified One-vs-Rest Kernel SVM from Scratch

We implement a simplified one-vs-rest kernel SVM using projected dual optimization.

This is intentionally restricted to a kernel subset because the kernel matrix has `O(n²)` memory cost.

In [ ]:
# ============================================================
# Section 35: Simplified Kernel SVM from Scratch
# ============================================================

class BinaryKernelSVM:
    def __init__(self, kernel="rbf", C=1.0, gamma=None, degree=2, coef0=1.0, lr=0.001, n_epochs=80):
        self.kernel = kernel
        self.C = C
        self.gamma = gamma
        self.degree = degree
        self.coef0 = coef0
        self.lr = lr
        self.n_epochs = n_epochs
    def fit(self, X, y_binary):
        self.X_train = np.asarray(X, dtype=float)
        y = np.where(np.asarray(y_binary) > 0, 1.0, -1.0)
        self.y = y
        K = compute_kernel(self.X_train, self.X_train, self.kernel, self.gamma, self.degree, self.coef0)
        Q = (y[:, None] * y[None, :]) * K
        alpha = np.zeros(len(y))
        for _ in range(self.n_epochs):
            grad = 1.0 - Q @ alpha
            alpha += self.lr * grad
            alpha = np.clip(alpha, 0.0, self.C)
        self.alpha = alpha
        return self
    def decision_function(self, X):
        K = compute_kernel(np.asarray(X, dtype=float), self.X_train, self.kernel, self.gamma, self.degree, self.coef0)
        return K @ (self.alpha * self.y)

class OneVsRestKernelSVM:
    def __init__(self, num_classes, kernel="rbf", C=1.0, gamma=None, degree=2, coef0=1.0, lr=0.001, n_epochs=80):
        self.num_classes = num_classes
        self.kernel = kernel
        self.C = C
        self.gamma = gamma
        self.degree = degree
        self.coef0 = coef0
        self.lr = lr
        self.n_epochs = n_epochs
        self.models = []
    def fit(self, X, y):
        self.models = []
        for c in range(self.num_classes):
            y_binary = (y == c).astype(int)
            model = BinaryKernelSVM(self.kernel, self.C, self.gamma, self.degree, self.coef0, self.lr, self.n_epochs)
            model.fit(X, y_binary)
            self.models.append(model)
        return self
    def predict(self, X):
        scores = np.column_stack([m.decision_function(X) for m in self.models])
        return np.argmax(scores, axis=1)

kernel_svm_results = []
for kernel_name in ["linear", "rbf"]:
    start = time.time()
    svm = OneVsRestKernelSVM(num_classes=num_classes, kernel=kernel_name, C=1.0, gamma=1.0 / X_train_complaint.shape[1], lr=0.001, n_epochs=80)
    svm.fit(X_train_complaint[kernel_complaint_idx], y_train_complaint[kernel_complaint_idx])
    pred = svm.predict(X_test_complaint[test_complaint_eval_idx])
    runtime = time.time() - start
    result = classification_metrics(y_test_complaint[test_complaint_eval_idx], pred, num_classes, label=f"Kernel SVM OvR {kernel_name}")
    result["runtime_seconds"] = runtime
    result["kernel"] = kernel_name
    kernel_svm_results.append(result)

kernel_svm_results_df = pd.DataFrame(kernel_svm_results)
display(kernel_svm_results_df)


## Section 36 — KPCA + Downstream Classifier

Kernel PCA creates nonlinear components using the centered kernel matrix.

We then train a downstream softmax classifier on the KPCA features.

In [ ]:
# ============================================================
# Section 36: KPCA + Downstream Classifier
# ============================================================

class KernelPCAFromScratch:
    def __init__(self, n_components=20, kernel="rbf", gamma=None, degree=2, coef0=1.0):
        self.n_components = n_components
        self.kernel = kernel
        self.gamma = gamma
        self.degree = degree
        self.coef0 = coef0
    def fit(self, X):
        self.X_fit = np.asarray(X, dtype=float)
        K = compute_kernel(self.X_fit, self.X_fit, self.kernel, self.gamma, self.degree, self.coef0)
        n = K.shape[0]
        one = np.ones((n, n)) / n
        self.K_fit_rows_mean = K.mean(axis=0)
        self.K_fit_total_mean = K.mean()
        Kc = K - one @ K - K @ one + one @ K @ one
        eigvals, eigvecs = np.linalg.eigh(Kc)
        order = np.argsort(eigvals)[::-1]
        eigvals = eigvals[order]
        eigvecs = eigvecs[:, order]
        positive = eigvals > 1e-10
        eigvals = eigvals[positive][:self.n_components]
        eigvecs = eigvecs[:, positive][:, :self.n_components]
        self.eigvals = eigvals
        self.alphas = eigvecs / np.sqrt(eigvals + 1e-12)
        return self
    def transform(self, X):
        K = compute_kernel(np.asarray(X, dtype=float), self.X_fit, self.kernel, self.gamma, self.degree, self.coef0)
        Kc = K - self.K_fit_rows_mean[None, :] - K.mean(axis=1)[:, None] + self.K_fit_total_mean
        return Kc @ self.alphas
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

start = time.time()
kpca = KernelPCAFromScratch(n_components=20, kernel="rbf", gamma=1.0 / X_train_complaint.shape[1])
X_kpca_train = kpca.fit_transform(X_train_complaint[kernel_complaint_idx])
X_kpca_test = kpca.transform(X_test_complaint[test_complaint_eval_idx])
kpca_clf = MulticlassLogisticRegressionGD(learning_rate=0.05, n_epochs=300, l2=1e-4, batch_size=256, random_state=RANDOM_STATE)
kpca_clf.fit(X_kpca_train, y_train_complaint[kernel_complaint_idx], num_classes=num_classes)
kpca_pred = kpca_clf.predict(X_kpca_test)
kpca_runtime = time.time() - start
kpca_result = classification_metrics(y_test_complaint[test_complaint_eval_idx], kpca_pred, num_classes, label="KPCA RBF + Logistic Regression")
kpca_result["runtime_seconds"] = kpca_runtime

display(pd.DataFrame([kpca_result]))


## Section 37 — Consolidated Results

We combine all non-kernel and kernel results into final result tables.

In [ ]:
# ============================================================
# Section 37: Consolidated Results
# ============================================================

regression_results = []
regression_results.append(linear_regression_result)
regression_results.append(knn_reg_result)
regression_results.append(tree_reg_result)
regression_results.extend(kernel_regression_results)
regression_results.extend(kernel_knn_results_reg)
regression_results_df = pd.DataFrame(regression_results)
regression_results_df = regression_results_df.sort_values("RMSE", ascending=True).reset_index(drop=True)

display(regression_results_df)

classification_results = []
classification_results.append(logistic_result)
classification_results.append(knn_clf_result)
classification_results.append(tree_clf_result)
classification_results.extend(kernel_knn_results_clf)
classification_results.extend(kernel_svm_results)
classification_results.append(kpca_result)
classification_results_df = pd.DataFrame(classification_results)
classification_results_df = classification_results_df.sort_values("macro_f1", ascending=False).reset_index(drop=True)

display(classification_results_df)

FINAL_DIR = OUTPUT_DIR / "final_results"
FINAL_DIR.mkdir(exist_ok=True)
regression_results_df.to_csv(FINAL_DIR / "nyc311_resolution_regression_results.csv", index=False)
classification_results_df.to_csv(FINAL_DIR / "nyc311_complaint_classification_results.csv", index=False)


## Section 38 — Kernel Investigation Discussion Table

The assignment requires mathematical and dataset-specific kernel explanations, not just raw metrics.

This table explains why each kernel may succeed or fail on NYC 311 data.

In [ ]:
# ============================================================
# Section 38: Kernel Investigation Discussion Table
# ============================================================

kernel_discussion_table = pd.DataFrame({
    "kernel": ["linear", "polynomial", "RBF"],
    "mathematical_behavior": [
        "Preserves linear similarity in the original engineered feature space.",
        "Captures finite-order interactions between variables such as borough, hour, channel, and complaint type.",
        "Creates localized similarity; nearby points in feature space strongly influence each other while distant points have low similarity."
    ],
    "expected_strength_on_nyc311": [
        "Useful when agency, borough, and temporal effects are approximately additive.",
        "Useful if interactions such as complaint type by hour or borough by channel affect resolution time.",
        "Useful for localized patterns such as specific borough-time-channel combinations and nonlinear geography effects."
    ],
    "expected_weakness_on_nyc311": [
        "Cannot represent strong nonlinear operational bottlenecks.",
        "Can become unstable or overfit sparse one-hot features if degree is too high.",
        "Computationally expensive and sensitive to gamma; kernel matrix memory is O(n²)."
    ]
})

display(kernel_discussion_table)
kernel_discussion_table.to_csv(FINAL_DIR / "nyc311_kernel_investigation_discussion.csv", index=False)


## Section 39 — Failure Analysis

The assignment requires isolating the 10 worst predictions and explaining why models failed.

We analyze:

- Worst resolution-time predictions
- Incorrect complaint category predictions
- Whether errors are caused by model limitation or data-quality artifacts

In [ ]:
# ============================================================
# Section 39: Failure Analysis
# ============================================================

# Use best available regression model by RMSE from stored predictions if available.
# Here we use linear regression predictions as a stable baseline for detailed row-level analysis.
failure_reg_idx = test_resolution_eval_idx
failure_reg_df = df_resolution_task.iloc[resolution_test_idx[failure_reg_idx]].copy()
failure_reg_pred_raw = pred_raw if len(pred_raw) == len(failure_reg_idx) else np.maximum(np.expm1(linreg.predict(X_test_resolution[failure_reg_idx])), 0)
failure_reg_df["true_resolution_time_hours"] = y_test_resolution_raw[failure_reg_idx]
failure_reg_df["predicted_resolution_time_hours"] = failure_reg_pred_raw
failure_reg_df["absolute_error_hours"] = np.abs(failure_reg_df["true_resolution_time_hours"] - failure_reg_df["predicted_resolution_time_hours"])

worst_resolution_predictions = failure_reg_df.sort_values("absolute_error_hours", ascending=False).head(10)
cols_to_show_reg = [col for col in [UNIQUE_KEY_COL, CREATED_DATE_COL, CLOSED_DATE_COL, CLASSIFICATION_TARGET_COL, "Problem Detail (formerly Descriptor)", "Agency", "Borough", "Incident Zip", "Open Data Channel Type", "Location Type", "near_duplicate_cluster_size", "high_confidence_outlier", "true_resolution_time_hours", "predicted_resolution_time_hours", "absolute_error_hours"] if col in worst_resolution_predictions.columns]
display(worst_resolution_predictions[cols_to_show_reg])

# Classification failure analysis using logistic regression baseline.
failure_clf_idx = test_complaint_eval_idx
failure_clf_df = df_complaint_task.iloc[complaint_test_idx[failure_clf_idx]].copy()
failure_clf_pred = pred_complaint if len(pred_complaint) == len(failure_clf_idx) else logreg.predict(X_test_complaint[failure_clf_idx])
failure_clf_df["true_label"] = y_test_complaint[failure_clf_idx]
failure_clf_df["predicted_label"] = failure_clf_pred
failure_clf_df["true_complaint_category"] = failure_clf_df["true_label"].map(index_to_class)
failure_clf_df["predicted_complaint_category"] = failure_clf_df["predicted_label"].map(index_to_class)
failure_clf_df["is_wrong"] = (failure_clf_df["true_label"] != failure_clf_df["predicted_label"]).astype(int)

wrong_complaints = failure_clf_df[failure_clf_df["is_wrong"] == 1].copy()
wrong_complaints["true_category_frequency_in_sample"] = wrong_complaints[CLASSIFICATION_TARGET_COL].map(df_complaint_task[CLASSIFICATION_TARGET_COL].value_counts())
worst_complaint_predictions = wrong_complaints.sort_values("true_category_frequency_in_sample", ascending=True).head(10)
cols_to_show_clf = [col for col in [UNIQUE_KEY_COL, CREATED_DATE_COL, CLASSIFICATION_TARGET_COL, "predicted_complaint_category", "Agency", "Borough", "Incident Zip", "Open Data Channel Type", "Location Type", "near_duplicate_cluster_size", "high_confidence_outlier"] if col in worst_complaint_predictions.columns]
display(worst_complaint_predictions[cols_to_show_clf])

failure_analysis_notes = pd.DataFrame({
    "failure_type": [
        "Extremely long resolution time",
        "Missing or unresolved Closed Date",
        "Near-duplicate reports",
        "Ambiguous complaint categories",
        "Sparse geographic categories",
        "Operational post-outcome leakage avoided"
    ],
    "interpretation": [
        "Some service requests remain open much longer than typical cases, creating heavy-tailed regression errors.",
        "Requests without valid closure cannot be used as regression labels and indicate label-quality limitations.",
        "Multiple reports for the same incident can distort category frequency and resolution estimates.",
        "Some categories are semantically close, so errors may reflect ambiguous ground truth rather than pure model failure.",
        "Rare ZIP codes or location types may have insufficient training examples.",
        "The model avoids closed-date and resolution fields, so performance is more realistic but lower than a leaked model."
    ],
    "systemic_or_data_quality": [
        "Both systemic operational limitation and data-quality challenge",
        "Data-quality and censoring issue",
        "Data collection artifact",
        "Label ambiguity",
        "Sampling sparsity / geographic bias",
        "Evaluation integrity decision"
    ]
})

display(failure_analysis_notes)
worst_resolution_predictions.to_csv(FINAL_DIR / "nyc311_worst_resolution_predictions.csv", index=False)
worst_complaint_predictions.to_csv(FINAL_DIR / "nyc311_worst_complaint_predictions.csv", index=False)
failure_analysis_notes.to_csv(FINAL_DIR / "nyc311_failure_analysis_notes.csv", index=False)


## Section 40 — Bias and Fairness Analysis

For NYC 311, model reliability may differ by:

- Borough
- ZIP code
- Reporting channel
- Time of day
- Complaint type

We analyze error by borough and channel.

In [ ]:
# ============================================================
# Section 40: Bias and Fairness Analysis
# ============================================================

# Regression error by borough
if "Borough" in failure_reg_df.columns:
    borough_reg_error = failure_reg_df.groupby("Borough").agg(
        n=("absolute_error_hours", "size"),
        mae_hours=("absolute_error_hours", "mean"),
        median_error_hours=("absolute_error_hours", "median"),
        true_resolution_median=("true_resolution_time_hours", "median")
    ).reset_index().sort_values("mae_hours", ascending=False)
    display(borough_reg_error)
    borough_reg_error.to_csv(FINAL_DIR / "nyc311_borough_resolution_error.csv", index=False)

# Classification accuracy by borough
if "Borough" in failure_clf_df.columns:
    borough_clf_error = failure_clf_df.groupby("Borough").agg(
        n=("is_wrong", "size"),
        error_rate=("is_wrong", "mean")
    ).reset_index()
    borough_clf_error["accuracy"] = 1 - borough_clf_error["error_rate"]
    borough_clf_error = borough_clf_error.sort_values("error_rate", ascending=False)
    display(borough_clf_error)
    borough_clf_error.to_csv(FINAL_DIR / "nyc311_borough_complaint_error.csv", index=False)

# Channel-level analysis
if "Open Data Channel Type" in failure_reg_df.columns:
    channel_reg_error = failure_reg_df.groupby("Open Data Channel Type").agg(
        n=("absolute_error_hours", "size"),
        mae_hours=("absolute_error_hours", "mean"),
        median_error_hours=("absolute_error_hours", "median")
    ).reset_index().sort_values("mae_hours", ascending=False)
    display(channel_reg_error)
    channel_reg_error.to_csv(FINAL_DIR / "nyc311_channel_resolution_error.csv", index=False)

bias_discussion_table = pd.DataFrame({
    "bias_type": ["Sampling bias", "Temporal drift", "Geographic bias", "Channel bias"],
    "nyc311_example": [
        "Some complaint categories dominate the dataset, so rare classes may be underrepresented.",
        "Complaint patterns and agency response times may change across years, seasons, or policy periods.",
        "Boroughs and ZIP codes may have different reporting rates and service response infrastructure.",
        "Online, mobile, and phone reports may represent different populations and complaint types."
    ],
    "effect_on_model_reliability": [
        "High performance on frequent categories may hide poor rare-category performance.",
        "Models trained on one period may not generalize to future operational conditions.",
        "Errors may be higher in underrepresented or operationally different areas.",
        "Predictions may reflect reporting behavior rather than true incident prevalence."
    ]
})

display(bias_discussion_table)
bias_discussion_table.to_csv(FINAL_DIR / "nyc311_bias_fairness_discussion.csv", index=False)


## Section 41 — Computational Analysis

Kernel methods require kernel matrices.

For `n` training samples, a full kernel matrix has shape `n × n`, which means:

- Memory complexity: `O(n²)`
- Matrix inversion/eigendecomposition: up to `O(n³)`

This is why we use controlled kernel subsets.

In [ ]:
# ============================================================
# Section 41: Computational Analysis
# ============================================================


def kernel_memory_mb(n, dtype_bytes=8):
    return (n * n * dtype_bytes) / (1024 ** 2)

computational_analysis_table = pd.DataFrame({
    "n_train_samples": [1_000, 2_000, 5_000, 10_000, 25_000, 50_000, len(X_train_complaint)],
    "kernel_matrix_memory_mb_float64": [kernel_memory_mb(n) for n in [1_000, 2_000, 5_000, 10_000, 25_000, 50_000, len(X_train_complaint)]],
    "kernel_matrix_memory_gb_float64": [kernel_memory_mb(n) / 1024 for n in [1_000, 2_000, 5_000, 10_000, 25_000, 50_000, len(X_train_complaint)]],
})

display(computational_analysis_table)

model_runtime_table = pd.concat([
    regression_results_df[["model", "runtime_seconds"]].assign(task="resolution_regression"),
    classification_results_df[["model", "runtime_seconds"]].assign(task="complaint_classification")
], axis=0, ignore_index=True)

display(model_runtime_table.sort_values("runtime_seconds", ascending=False))

computational_analysis_table.to_csv(FINAL_DIR / "nyc311_kernel_memory_analysis.csv", index=False)
model_runtime_table.to_csv(FINAL_DIR / "nyc311_model_runtime_analysis.csv", index=False)


## Section 42 — Theory vs Reality Discussion

Textbook datasets usually underestimate the difficulty of production machine learning.

NYC 311 shows why real-world datasets are harder.

In [ ]:
# ============================================================
# Section 42: Theory vs Reality Discussion
# ============================================================

theory_vs_reality_table = pd.DataFrame({
    "textbook_assumption": [
        "Clean labels",
        "Small and balanced classes",
        "No missing values",
        "Independent observations",
        "Stable distribution",
        "No leakage risk",
        "Low computational cost"
    ],
    "nyc311_reality": [
        "Complaint categories can be ambiguous and operationally assigned.",
        "Some complaint types dominate while many classes are rare.",
        "Closed Date, address, coordinates, and resolution fields may be missing.",
        "Near-duplicate reports can describe the same real-world incident.",
        "Complaint volume and resolution behavior vary by time, borough, and policy conditions.",
        "Closed Date and resolution fields can trivially leak the regression target.",
        "Kernel matrices become infeasible on the full dataset."
    ],
    "engineering_response": [
        "Perform label-quality analysis and failure analysis.",
        "Use stratified sampling and macro-averaged classification metrics.",
        "Classify missingness and use robust imputation strategies.",
        "Detect exact and near-duplicate reports.",
        "Engineer temporal features and discuss temporal drift.",
        "Build task-specific leakage filters.",
        "Use controlled subsets for kernel methods and report memory complexity."
    ]
})

display(theory_vs_reality_table)
theory_vs_reality_table.to_csv(FINAL_DIR / "nyc311_theory_vs_reality_discussion.csv", index=False)


## Section 43 — Final Report-Ready Summary for Dataset 2

This section provides a concise report-ready summary for the NYC 311 dataset.

The NYC 311 Service Requests dataset required a different engineering strategy from the Airbnb dataset. The main challenge was scale, class imbalance, duplicate reports, temporal structure, and post-outcome leakage risk.

An intelligent stratified sample was constructed to preserve the complaint-category distribution while keeping the dataset computationally manageable. Exact duplicate records were removed using the `Unique Key`, and near-duplicate reports were identified using complaint type, descriptor, address, borough, ZIP code, and creation-time window.

The classification target was the complaint category. The regression target was resolution time in hours, calculated from `Closed Date - Created Date`. Missing or invalid resolution-time labels were documented and excluded from supervised regression modeling.

Temporal features were engineered from `Created Date`, including hour, day of week, month, weekend flag, after-hours flag, season, and cyclical encodings. Post-outcome fields such as `Closed Date`, `Status`, and `Resolution Description` were excluded from predictive features to avoid leakage.

Missingness was investigated using missingness indicators and statistical association tests. Many missing values were consistent with MAR, especially fields related to closure, location, and reporting behavior. Some address-related missingness was marked as MAR / suspected MNAR because missingness may depend on unobserved reporting uncertainty.

Outliers were analyzed using Z-score, IQR, and a custom NumPy Isolation Forest. Resolution time showed strong skewness, so outlier handling focused on documentation, log transformation, and high-confidence outlier flags rather than blind deletion.

Non-kernel models and kernel models were implemented from scratch. Kernel methods were evaluated on controlled subsets due to the `O(n²)` memory and `O(n³)` computational bottlenecks of kernel matrices.

The final analysis includes model performance, kernel-specific explanations, failure analysis, bias discussion by borough and channel, computational analysis, and theory-vs-reality discussion.

## Section 44 — Save Final Cleaned Dataset Snapshot

We save the final analytical sample and key tables.

In [ ]:
# ============================================================
# Section 44: Save Final Cleaned Dataset Snapshot
# ============================================================

nyc311.to_csv(OUTPUT_DIR / "nyc311_final_analytical_sample.csv", index=False)
df_complaint_task.to_csv(OUTPUT_DIR / "nyc311_complaint_task_dataframe.csv", index=False)
df_resolution_task.to_csv(OUTPUT_DIR / "nyc311_resolution_task_dataframe.csv", index=False)

print("Final NYC 311 outputs saved.")
print("Main output directory:", OUTPUT_DIR.resolve())
print("Final results directory:", FINAL_DIR.resolve())


completion_marker = pd.DataFrame({
    "status": ["core_pipeline_completed"],
    "dataset": ["NYC 311 Service Requests"],
    "note": ["Core analytical, modeling, failure-analysis, bias, and computational sections completed before final audit sections."]
})
completion_marker.to_csv(FINAL_DIR / "nyc311_core_pipeline_completion_marker.csv", index=False)


## Section 45 — Final Assignment Compliance Checklist

This section verifies that the NYC 311 notebook covers the dataset-specific requirements from the assignment: intelligent sampling, duplicate handling, temporal features, complaint category classification, resolution-time regression, data-quality investigation, outlier analysis, leakage-safe feature selection, kernel/non-kernel models, failure analysis, bias discussion, computational analysis, and report-ready interpretation.

In [ ]:
# ============================================================
# Section 45: Final Assignment Compliance Checklist
# ============================================================

assignment_compliance_items = [
    ("Dataset loaded from ZIP", "Sections 1-2", "done"),
    ("Complaint category classification target", "Sections 3 and 23", "done"),
    ("Resolution time regression target", "Section 8", "done"),
    ("Intelligent stratified sampling", "Sections 4-6", "done"),
    ("Duplicate and near-duplicate report handling", "Section 7", "done"),
    ("Temporal feature engineering", "Section 9", "done"),
    ("Geographic and channel feature engineering", "Section 10", "done"),
    ("Missing data analysis and missingness classification", "Sections 11-14", "done"),
    ("Outlier comparison: Z-score, IQR, Simple Isolation Forest", "Sections 15-18", "done"),
    ("Feature quality and leakage analysis", "Sections 19-22", "done"),
    ("Train/test split from scratch", "Section 24", "done"),
    ("Preprocessing from scratch", "Section 25", "done"),
    ("Non-kernel models from scratch", "Sections 29-32", "done"),
    ("Kernel methods from scratch", "Sections 33-36", "done"),
    ("Kernel investigation discussion", "Section 38", "done"),
    ("Failure analysis", "Section 39", "done"),
    ("Bias and fairness analysis", "Section 40", "done"),
    ("Computational complexity analysis", "Section 41", "done"),
    ("Theory vs reality discussion", "Section 42", "done"),
    ("Report-ready dataset summary", "Section 43", "done"),
]

nyc311_assignment_compliance_table = pd.DataFrame(assignment_compliance_items, columns=["requirement", "notebook_location", "status"])
display(nyc311_assignment_compliance_table)
nyc311_assignment_compliance_table.to_csv(FINAL_DIR / "nyc311_assignment_compliance_checklist.csv", index=False)


## Section 46 — Final Leakage and Reproducibility Audit

This section performs a final defensive audit. It verifies that direct targets, post-outcome fields, and global outlier-analysis artifacts are not used as model inputs. It also checks that the final matrices and result tables exist and contain valid values.

In [ ]:
# ============================================================
# Section 46: Final Leakage and Reproducibility Audit
# ============================================================

final_forbidden_columns = set([
    CLASSIFICATION_TARGET_COL,
    REGRESSION_TARGET_COL,
    UNIQUE_KEY_COL,
    CLOSED_DATE_COL,
    "closed_datetime",
    "Status",
    "Due Date",
    "Resolution Description",
    "Resolution Action Updated Date",
    "valid_resolution_time_label",
    "target_missing_created_date",
    "target_missing_closed_date",
    "target_negative_resolution_time",
    "target_extreme_resolution_time_gt_365_days",
    "simple_iforest_score",
    "simple_iforest_outlier",
    "zscore_outlier_any",
    "iqr_outlier_any",
    "number_of_outlier_methods_flagged",
    "high_confidence_outlier",
])

complaint_leakage_violations = sorted(set(complaint_candidate_features).intersection(final_forbidden_columns))
resolution_leakage_violations = sorted(set(resolution_candidate_features).intersection(final_forbidden_columns))

if complaint_leakage_violations or resolution_leakage_violations:
    raise ValueError({
        "complaint_leakage_violations": complaint_leakage_violations,
        "resolution_leakage_violations": resolution_leakage_violations,
    })

required_runtime_objects = {
    "X_train_complaint": X_train_complaint,
    "X_test_complaint": X_test_complaint,
    "y_train_complaint": y_train_complaint,
    "y_test_complaint": y_test_complaint,
    "X_train_resolution": X_train_resolution,
    "X_test_resolution": X_test_resolution,
    "y_train_resolution_log": y_train_resolution_log,
    "y_test_resolution_log": y_test_resolution_log,
    "regression_results_df": regression_results_df,
    "classification_results_df": classification_results_df,
}

audit_rows = []
for name, obj in required_runtime_objects.items():
    if isinstance(obj, pd.DataFrame):
        audit_rows.append({"object": name, "status": "available", "shape": str(obj.shape)})
    else:
        audit_rows.append({"object": name, "status": "available", "shape": str(np.shape(obj))})

# Matrix integrity checks
for name, X in [("X_train_complaint", X_train_complaint), ("X_test_complaint", X_test_complaint), ("X_train_resolution", X_train_resolution), ("X_test_resolution", X_test_resolution)]:
    if np.isnan(X).any() or np.isinf(X).any():
        raise ValueError(f"{name} contains NaN or infinite values.")

if "RMSE" not in regression_results_df.columns:
    raise ValueError(f"Regression results must contain RMSE. Columns: {regression_results_df.columns.tolist()}")
if "macro_f1" not in classification_results_df.columns:
    raise ValueError(f"Classification results must contain macro_f1. Columns: {classification_results_df.columns.tolist()}")

nyc311_final_audit_table = pd.DataFrame(audit_rows)
display(nyc311_final_audit_table)
nyc311_final_audit_table.to_csv(FINAL_DIR / "nyc311_final_leakage_reproducibility_audit.csv", index=False)

print("Final leakage and reproducibility audit passed.")


## Section 47 — Final Model Selection and Interpretation Tables

This section selects the best regression model by lowest RMSE and the best classification model by highest macro-F1. It also creates report-ready interpretation points for the NYC 311 dataset.

In [ ]:
# ============================================================
# Section 47: Final Model Selection and Interpretation Tables
# ============================================================

best_resolution_model = regression_results_df.sort_values("RMSE", ascending=True).head(1).copy()
best_complaint_model = classification_results_df.sort_values("macro_f1", ascending=False).head(1).copy()

final_model_selection_table = pd.concat([
    best_resolution_model.assign(final_task="resolution_time_regression"),
    best_complaint_model.assign(final_task="complaint_category_classification")
], ignore_index=True, sort=False)

nyc311_interpretation_points = pd.DataFrame({
    "finding": [
        "Complaint classes are operational and imbalanced",
        "Resolution time is highly skewed",
        "Temporal and geographic context matters",
        "Duplicate and near-duplicate reports affect reliability",
        "Kernel methods require subsampling"
    ],
    "interpretation": [
        "Frequent complaint categories dominate accuracy, so macro-F1 is important for rare classes.",
        "Some requests take much longer than typical cases, making RMSE sensitive to extreme errors.",
        "Hour, day, borough, ZIP, and reporting channel reflect operational patterns and service availability.",
        "Repeated reports of the same incident may bias learning if not explicitly detected.",
        "The full NYC 311 dataset is too large for naive full kernel matrices, so controlled subsampling is necessary."
    ]
})

display(final_model_selection_table)
display(nyc311_interpretation_points)

final_model_selection_table.to_csv(FINAL_DIR / "nyc311_final_model_selection_table.csv", index=False)
nyc311_interpretation_points.to_csv(FINAL_DIR / "nyc311_interpretation_points.csv", index=False)


## Section 48 — Mini-Conference Paper Section for Dataset 2

The following text is written in report-ready form and can be adapted into the final mini-conference paper.

In [ ]:
# ============================================================
# Section 48: Mini-Conference Paper Section for Dataset 2
# ============================================================

nyc311_report_section = """
Dataset 2: NYC 311 Service Requests

The NYC 311 dataset represents a large-scale municipal service-request system. Unlike small textbook datasets, it contains operational labels, temporal drift, missing fields, duplicate and near-duplicate reports, geographic heterogeneity, and severe class imbalance across complaint categories. The project therefore uses an engineering-first pipeline before model training.

The complaint category prediction task is formulated as a multiclass classification problem using the top complaint categories. Because the raw dataset is large, an intelligent stratified sampling strategy is used to preserve the relative distribution of selected complaint types while keeping computation feasible. Sampling integrity is verified by comparing the original and sampled category distributions.

The resolution time prediction task is formulated as a regression problem. The target is engineered from the difference between Closed Date and Created Date. Invalid labels, including missing dates, negative durations, and extremely long durations beyond one year, are explicitly identified and excluded from supervised regression. A log-transformed resolution target is used for model training because the raw resolution-time distribution is highly right-skewed.

Data-quality investigation includes missingness analysis, duplicate handling, near-duplicate report detection, outlier analysis using Z-score, IQR, and a custom Simple Isolation Forest from scratch, and feature-quality analysis. Temporal features such as hour, day of week, month, weekend status, after-hours status, and cyclic encodings are engineered from Created Date. Geographic and reporting-channel features are also included when available.

Leakage control is task-specific. Closed Date, status, resolution descriptions, direct resolution-time variables, and target-derived fields are excluded from model inputs. Global outlier-analysis artifacts are also excluded from predictive features because they are computed before train/test splitting and may encode distribution-level information.

Non-kernel methods and kernel methods are implemented from scratch using NumPy. Non-kernel models include linear regression, multiclass logistic regression, KNN, and decision trees. Kernel methods include kernel ridge regression, kernel KNN, a simplified one-vs-rest kernel SVM, and KPCA followed by logistic regression. Kernel experiments are performed on controlled subsets because kernel matrix memory grows quadratically with the number of samples.

Failure analysis shows that the largest resolution-time errors often correspond to unusually long service delays, ambiguous complaint categories, or geographically/channel-specific operational patterns. Bias and fairness analysis examines borough-level and channel-level error differences, showing that aggregate performance can hide uneven reliability across communities and reporting channels.
"""

print(nyc311_report_section)
with open(FINAL_DIR / "nyc311_report_ready_section.txt", "w", encoding="utf-8") as f:
    f.write(nyc311_report_section)


## Section 49 — Final Deliverable Package Index

This section records the final important output files generated by the NYC 311 notebook.

In [ ]:
# ============================================================
# Section 49: Final Deliverable Package Index
# ============================================================

final_output_files = []
for folder in [OUTPUT_DIR, FINAL_DIR, MODELING_DIR]:
    if folder.exists():
        for path in sorted(folder.glob("*")):
            if path.is_file():
                final_output_files.append({
                    "folder": str(folder),
                    "file_name": path.name,
                    "size_kb": path.stat().st_size / 1024
                })

nyc311_final_deliverable_index = pd.DataFrame(final_output_files).sort_values(["folder", "file_name"]).reset_index(drop=True)
display(nyc311_final_deliverable_index)
nyc311_final_deliverable_index.to_csv(FINAL_DIR / "nyc311_final_deliverable_index.csv", index=False)

print("Dataset 2 NYC 311 notebook is complete and audit-ready.")
